# Dependency

In [10]:
# !pip install ollama rapidfuzz
# !pip install diffusers transformers accelerate torch torchvision deepface opencv-python scipy
# !pip install tf-keras
# !pip install -q ipywidgets TQDM


In [4]:
# import ollama
import json
import random
# from rapidfuzz import fuzz
import time
import os
import torch
# from deepface import DeepFace
# from scipy.spatial.distance import cosine
# import cv2
# import numpy as np
# from PIL import Image, ImageDraw, ImageFont
# from huggingface_hub import hf_hub_download
# from diffusers import StableDiffusionXLPipeline



# # --- PENGECEKAN AKHIR DEPENDENSI ---
# required_libraries = [
#     ('ollama', 'ollama'),
#     ('json', 'json'),
#     ('random', 'random'),
#     ('rapidfuzz', 'rapidfuzz'),
#     ('time', 'time'),
#     ('os', 'os'),
#     ('torch', 'torch'),
#     ('diffusers', 'diffusers'),
#     ('deepface', 'deepface'),
#     ('scipy', 'scipy'),
#     ('cv2', 'cv2'),
#     ('numpy', 'numpy'),
#     ('PIL', 'Pillow'),
#     ('huggingface_hub', 'huggingface_hub'),
# ]

# missing_libraries = []

# for module_name, package_name in required_libraries:
#     try:
#         __import__(module_name)
#     except ImportError:
#         missing_libraries.append(package_name)

# if missing_libraries:
#     print("\n[EROR] Beberapa library belum terinstal:")
#     for lib in missing_libraries:
#         print(f"  - {lib}")
#     print("\nSilakan instal dengan perintah berikut:")
#     print(f"pip install {' '.join(missing_libraries)}")
#     exit(1)
# else:
#     print("\n[SUKSES] Semua library siap digunakan!")

# print(f"CUDA Available: {torch.cuda.is_available()}")
# print(f"GPU Name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# Configuration

In [5]:
# ==========================================
# 1. MASTER DIRECTORY (SISTEM ESTAFET)
# ==========================================
DIR_FASE1 = "Fase1-Output"
DIR_FASE2 = "Fase2-Output"
DIR_FASE3_1 = "Fase3.1-Output"
DIR_FASE3 = "Fase3-Output"
DIR_FASE4 = "Fase4-Output"
DIR_FASE5 = "Fase5-Output"
DIR_FASE6 = "Fase6-Output"
DIR_FASE7 = "Fase7-Output"

# Path original template milikmu
TEMPLATE_DIR = r"D:\Exp_Fathur\GenKTP\Fake-EKTP-Generator-With-Local-LLM-GAN-For-Data-Generator\Template"

# Buat semua folder otomatis jika belum ada
for d in [DIR_FASE1, DIR_FASE2, DIR_FASE3, DIR_FASE3_1, DIR_FASE4, DIR_FASE5, DIR_FASE6, DIR_FASE7]:
    os.makedirs(d, exist_ok=True)

# ==========================================
# 2. CHECKPOINT FILES (FILE JSON PENGHUBUNG)
# ==========================================
FILE_FASE1 = os.path.join(DIR_FASE1, "data_fase1.json")
FILE_FASE2 = os.path.join(DIR_FASE2, "data_fase2.json")
FILE_FASE3 = os.path.join(DIR_FASE3, "config_fase3.json")
FILE_FASE3_1 = os.path.join(DIR_FASE3_1, "data_fase3_1.json")
FILE_FASE4 = os.path.join(DIR_FASE4, "data_fase4.json")
FILE_FASE5 = os.path.join(DIR_FASE5, "data_fase5.json")
FILE_FASE6 = os.path.join(DIR_FASE6, "data_fase6.json")
FILE_GROUND_TRUTH = os.path.join(DIR_FASE7, "ground_truth_labels.json")

# ==========================================
# 3. GLOBAL PARAMETERS (PARAMETER MODEL & AI)
# ==========================================

# --- Parameter Fase 1 (LLM Text) ---
TARGET_TOTAL = 100
BATCH_SIZE = 20 
OLLAMA_MODEL = "qwen2.5:7b" 
FUZZY_THRESHOLD = 75 

# Variabel State untuk Fase 1
final_datasets = []
existing_niks = set() 
existing_names = [] 
male_count = 0
female_count = 0
duplication_attempts = 0

# --- Parameter Fase 2 (Wajah AI) ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "SG161222/Realistic_Vision_V6.0_B1_noVAE"
EMBEDDING_MODEL = "Facenet"
DISTANCE_THRESHOLD = 0.55 
accepted_embeddings = []

# --- Parameter Fase 5 (Warping Resolusi KTP) ---
CANVAS_W, CANVAS_H = 856, 540 

print("✅ Master Configuration berhasil dimuat! Seluruh pipeline sudah terhubung.")

✅ Master Configuration berhasil dimuat! Seluruh pipeline sudah terhubung.


# Fase 1 - Unique Text Data Generator

In [13]:
def get_ollama_prompt(gender, count):
    """Membuat prompt spesifik dengan aturan KETAT agar data identitas realistis."""
    return f"""
    Buatkan {count} data identitas fiktif KTP Indonesia untuk jenis kelamin {gender}.
    Format output HARUS dalam JSON array murni. JANGAN ada teks pengantar.
    
    ATURAN KETAT (Patuhi atau Gagal):
    1. "provinsi": Gunakan nama provinsi asli di Indonesia (misal: "PROVINSI JAWA BARAT").
    2. "kota_kab": Awali dengan "KOTA " atau "KABUPATEN ".
    3. "nik": 16 digit angka acak yang realistis (tidak boleh berurutan seperti 123456789).
    4. "nama": Nama orang Indonesia asli (huruf kapital).
    5. "tempat_lahir": HANYA NAMA KOTA (huruf kapital).
    6. "tgl_lahir": Format DD-MM-YYYY.
    7. "jenis_kelamin": "{gender}".
    8. "gol_darah": HANYA PILIH: "A", "B", "AB", "O", atau "-".
    9. "alamat": Gunakan awalan "JL." atau "KP." (JANGAN GUNAKAN KATA "KAMPUNGAN"). Contoh: "JL. MERDEKA NO. 15" atau "KP. DURIAN RT 01".
    10. "rt_rw": HARUS format 3 angka garis miring 3 angka (TIDAK BOLEH ADA HURUF). Contoh: "001/003", "012/005".
    11. "kel_desa": NAMA KELURAHAN SAJA (JANGAN TULIS KATA "KELURAHAN" atau "DESA").
    12. "kecamatan": NAMA KECAMATAN SAJA (JANGAN TULIS KATA "KECAMATAN").
    13. "agama": HANYA PILIH: "ISLAM", "KRISTEN", "KATHOLIK", "HINDU", "BUDHA", "KONGHUCU".
    14. "status_perkawinan": HANYA PILIH: "BELUM KAWIN", "KAWIN", "CERAI HIDUP", "CERAI MATI".
    15. "pekerjaan": HANYA PILIH DARI DAFTAR INI: "PELAJAR/MAHASISWA", "MENGURUS RUMAH TANGGA", "WIRASWASTA", "KARYAWAN SWASTA", "PEGAWAI NEGERI SIPIL", "BURUH HARIAN LEPAS", "PETANI/PEKEBUN", "TENTARA NASIONAL INDONESIA", "KEPOLISIAN RI". JANGAN MENGARANG PEKERJAAN LAIN!
    16. "kewarganegaraan": "WNI".
    17. "berlaku_hingga": "SEUMUR HIDUP".
    18. "tgl_pembuatan": Format DD-MM-YYYY (Tahun antara 2018 - 2023).
    """

def is_name_unique(new_name, names_list, threshold):
    """Mengecek kemiripan nama menggunakan Levenshtein Distance."""
    if not names_list:
        return True
    
    # Bandingkan dengan semua nama yang sudah ada
    for existing_name in names_list:
        # Menghitung skor kemiripan (0-100)
        similarity_score = fuzz.ratio(new_name.lower(), existing_name.lower())
        if similarity_score >= threshold:
            return False # Terlalu mirip
            
    return True # Cukup unik

# ==========================================
# LOOP UTAMA PEMBANGKITAN DATA (MAIN)
# ==========================================
start_time = time.time()
print(f"🚀 Memulai Fase 1: Membangkitkan {TARGET_TOTAL} data KTP lengkap menggunakan model '{OLLAMA_MODEL}'...")

while len(final_datasets) < TARGET_TOTAL:
    # 1. Tentukan Gender berdasarkan kuota (50/50)
    if male_count < (TARGET_TOTAL / 2):
        current_gender_request = "LAKI-LAKI"
    else:
        current_gender_request = "PEREMPUAN"
        
    current_count_needed = TARGET_TOTAL - len(final_datasets)
    # Jangan meminta lebih dari batch size
    request_amount = min(BATCH_SIZE, current_count_needed) 

    print(f"--- Meminta batch {request_amount} data {current_gender_request} ke Ollama... ---")

    try:
        # 2. Panggil API Ollama
        response = ollama.chat(model=OLLAMA_MODEL, messages=[
            {
                'role': 'system',
                'content': 'You are a raw JSON data generator API. You output ONLY valid JSON arrays. Do not text outside the JSON.'
            },
            {
                'role': 'user',
                'content': get_ollama_prompt(current_gender_request, request_amount)
            },
        ])
            
        # 3. Parsing JSON hasil Ollama
        raw_content = response['message']['content'].strip()
            
        # [UPDATE] Ekstraksi Paksa Array JSON
        start_idx = raw_content.find('[')
        end_idx = raw_content.rfind(']')
            
        if start_idx != -1 and end_idx != -1:
            # Potong string hanya dari '[' sampai ']'
            clean_json_str = raw_content[start_idx:end_idx+1]
            generated_batch = json.loads(clean_json_str)
        else:
            # Jika sama sekali tidak ada kurung siku, kita log output aslinya untuk di-debug
            print(f"⚠️ DEBUG OUTPUT LLM: {raw_content[:200]}...") 
            raise json.JSONDecodeError("Array JSON tidak ditemukan", raw_content, 0)

        # 4. Validasi dan Filter Duplikasi
        valid_entries_in_batch = 0
        for entry in generated_batch:
            nik = str(entry.get('nik', ''))
            nama = entry.get('nama', '').strip()
            
            # Cek Validitas Dasar (pastikan NIK 16 digit dan LLM memberikan field penting)
            if len(nik) != 16 or not nama or "provinsi" not in entry:
                duplication_attempts += 1
                continue

            # FILTER 1: Cek NIK Ganda (Exact Match)
            if nik in existing_niks:
                print(f"   ⚠️ NIK Duplikat ditolak: {nik}")
                duplication_attempts += 1
                continue
                
            # FILTER 2: Cek Kemiripan Nama (Fuzzy Match)
            if not is_name_unique(nama, existing_names, FUZZY_THRESHOLD):
                print(f"   ⚠️ Nama Terlalu Mirip ditolak: {nama}")
                duplication_attempts += 1
                continue

            # FILTER 3: Pembersihan Teks Otomatis (Auto-Correct)
            # Hapus kata KAMPUNGAN jadi KP.
            entry['alamat'] = str(entry.get('alamat', '')).replace("KAMPUNGAN", "KP.").replace("kampungan", "KP.")
            
            # Hapus kata KELURAHAN/DESA/KECAMATAN yang berlebih
            entry['kel_desa'] = str(entry.get('kel_desa', '')).replace("KELURAHAN ", "").replace("DESA ", "")
            entry['kecamatan'] = str(entry.get('kecamatan', '')).replace("KECAMATAN ", "")
            
            # Paksa RT/RW hanya angka (jika LLM nakal memberi huruf)
            rt_rw_raw = str(entry.get('rt_rw', '001/001'))
            clean_rt_rw = ''.join(c for c in rt_rw_raw if c.isdigit() or c == '/')
            # Format paksa jika gagal
            if len(clean_rt_rw) < 7 or "/" not in clean_rt_rw:
                clean_rt_rw = f"{random.randint(1,15):03d}/{random.randint(1,15):03d}"
            entry['rt_rw'] = clean_rt_rw
            
            # Pastikan semua field berupa string untuk menghindari error di Fase 5
            for key in entry:
                if entry[key] is None:
                    entry[key] = ""
                else:
                    entry[key] = str(entry[key])

            # Jika lolos semua filter, simpan
            final_datasets.append(entry)
            existing_niks.add(nik)
            existing_names.append(nama)
            valid_entries_in_batch += 1
            
            # Update statistik gender
            if current_gender_request == "LAKI-LAKI":
                male_count += 1
            else:
                female_count += 1
                
        print(f"✅ Berhasil menambahkan {valid_entries_in_batch} data unik dari batch ini.")
        print(f"📈 Progress: {len(final_datasets)}/{TARGET_TOTAL} (L:{male_count}, P:{female_count})")

    except json.JSONDecodeError:
        print("❌ Error: Ollama tidak mengembalikan JSON yang valid. Mencoba lagi...")
        duplication_attempts += request_amount
    except Exception as e:
        print(f"❌ Error Tak Terduga: {e}")
        time.sleep(2) # Beri jeda jika error sistem

end_time = time.time()

# ==========================================
# FINALISASI & VERIFIKASI
# ==========================================
print("\n" + "="*50)
print("✅ FASE 1 SELESAI")
print("="*50)
print(f"Total Data Unik Berhasil Dibuat: {len(final_datasets)}")
print(f"Komposisi Gender: L={male_count}, P={female_count}")
print(f"Total Data Ditolak (Duplikat/Bad Format): {duplication_attempts}")
print(f"Waktu Eksekusi: {end_time - start_time:.2f} detik")
print("="*50)

# Tampilkan 1 contoh data teratas untuk memverifikasi kelengkapan field
print("\nContoh struktur data yang dihasilkan:")
print(json.dumps(final_datasets[0], indent=2))

# ==========================================
#  PENYIMPANAN KE DISK - FASE 1
# ==========================================
# --- SIMPAN FASE 1 ---
with open(FILE_FASE1, "w", encoding="utf-8") as f:
    json.dump(final_datasets, f, indent=4, ensure_ascii=False)
print(f"✅ Data teks Fase 1 tersimpan di {FILE_FASE1}")

🚀 Memulai Fase 1: Membangkitkan 100 data KTP lengkap menggunakan model 'qwen2.5:7b'...

✅ FASE 1 SELESAI
Total Data Unik Berhasil Dibuat: 100
Komposisi Gender: L=51, P=49
Total Data Ditolak (Duplikat/Bad Format): 1076
Waktu Eksekusi: 0.00 detik

Contoh struktur data yang dihasilkan:
{
  "provinsi": "PROVINSI JAWA BARAT",
  "kota_kab": "KOTA BANDUNG",
  "nik": "0123456789101112",
  "nama": "MUHAMIAD SUDARSONO",
  "tempat_lahir": "BANDUNG",
  "tgl_lahir": "15-10-1987",
  "jenis_kelamin": "LAKI-LAKI",
  "gol_darah": "A",
  "alamat": "JL. CIREBON RAYA NO. 34",
  "rt_rw": "021/005",
  "kel_desa": "CIGANDU",
  "kecamatan": "CIPEMBAH",
  "agama": "ISLAM",
  "status_perkawinan": "KAWIN",
  "pekerjaan": "PEGAWAI NEGERI SIPIL",
  "kewarganegaraan": "WNI",
  "berlaku_hingga": "SEUMUR HIDUP",
  "tgl_pembuatan": "15-03-2023"
}
✅ Data teks Fase 1 tersimpan di Fase1-Output\data_fase1.json


# Fase 2 - Unique Potrait Image Generator

In [41]:


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_ID = "SG161222/RealVisXL_V4.0"

print("⏳ Mengunduh dan memuat model RealVisXL (Arsitektur SDXL)...")

# Menggunakan StableDiffusionXLPipeline (Bukan pipeline biasa)
# variant="fp16" memastikan kita mendownload versi ringan dan cepat untuk RTX 40-series
pipe = StableDiffusionXLPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    variant="fp16",
    use_safetensors=True
)

# Optimasi Khusus VRAM 12GB (RTX 4070 Ti) agar tidak Out of Memory
pipe.enable_model_cpu_offload()

try:
    pipe.enable_xformers_memory_efficient_attention()
except Exception:
    pass

print("✅ Model SDXL Super-Realistik siap digunakan!")

⏳ Mengunduh dan memuat model RealVisXL (Arsitektur SDXL)...


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

✅ Model SDXL Super-Realistik siap digunakan!


In [42]:


# ==========================================
# 1. FUNGSI PROMPT KHUSUS SDXL (Lebih Natural & Akurat)
# ==========================================
def get_sdxl_prompt(gender, birth_year):
    try:
        year_int = int(birth_year[-4:])
        bg_color = "pure blue" if year_int % 2 == 0 else "pure red"
        age = 2026 - year_int
    except Exception:
        bg_color = random.choice(["pure blue", "pure red"])
        age = random.choice([20, 30, 40, 50])

    # HACK SDXL: Gunakan NAMA ACAK untuk memaksa struktur wajah yang berbeda!
    male_names = ["Budi", "Agus", "Hassan", "Joko", "Wawan", "Rizky", "Yusuf", "Hendra"]
    female_names = ["Siti", "Ratna", "Dewi", "Ayu", "Rini", "Sari", "Lestari", "Putri"]
    
    eth = random.choice(["Javanese", "Sundanese", "Bataknese", "Malay"])
    st = random.choice(["brown skin", "tanned skin"])
    jaw = random.choice(["sharp jawline", "round face", "chubby face", "oval face"])
    expr = random.choice(["neutral expression", "slight smile", "serious face", "relaxed face"])

    if gender == "LAKI-LAKI":
        name = random.choice(male_names)
        subject = f"{age}yo {eth} man named {name}, {st}, {jaw}"
        hs = random.choice(["short black hair", "comb over", "buzz cut"])
        ex = random.choice(["clean shaven", "thin mustache", "wearing glasses"])
    else:
        name = random.choice(female_names)
        subject = f"{age}yo {eth} woman named {name}, {st}, {jaw}"
        hs = random.choice(["tied black hair", "black hijab", "navy hijab"])
        ex = random.choice(["minimal makeup", "wearing glasses", "no glasses"])

    # PROMPT SANGAT PENDEK (Bebas dari limit 77 Token)
    prompt = f"RAW passport photo of {subject}, {expr}, {hs}, {ex}, wearing formal white shirt, solid {bg_color} background, front view, flat studio lighting"

    negative_prompt = "hands, fingers, tie, casual clothes, shadows, textured background, outdoor, smiling with teeth, deformed, anime"

    return prompt, negative_prompt

# ==========================================
def verify_face_quality(image_path, embeddings_list, threshold):
    try:
        results = DeepFace.represent(img_path=image_path, model_name=EMBEDDING_MODEL, enforce_detection=True)
        if len(results) > 1: return False, None, f"Ditolak: Terdeteksi {len(results)} wajah."
            
        face_data = results[0]
        new_embedding = face_data["embedding"]
        facial_area = face_data["facial_area"] 
        
        # PERBAIKAN: Kembalikan ke 85px. 
        # Pada ukuran gambar 300x400, wajah 85-120px itu proporsi pas foto yang normal.
        if facial_area['w'] < 85 or facial_area['h'] < 85:
            return False, None, f"Ditolak: Wajah terlalu kecil ({facial_area['w']}x{facial_area['h']})."
            
    except Exception:
        return False, None, "Ditolak: Wajah tidak ditemukan."

    for idx, ext_emb in enumerate(embeddings_list):
        dist = cosine(new_embedding, ext_emb)
        if dist < threshold:
            return False, None, f"Ditolak: Mirip orang ke-{idx} (Jarak {dist:.2f})"

    return True, new_embedding, "Valid"

# ==========================================
# 3. LOAD DATA & SINKRONISASI
# ==========================================
source_file = FILE_FASE2 if os.path.exists(FILE_FASE2) else FILE_FASE1
with open(source_file, "r", encoding="utf-8") as f:
    final_datasets = json.load(f)

print("\n🔄 Melakukan Sinkronisasi Folder & Memori DeepFace...")
accepted_embeddings = []

for i, data in enumerate(final_datasets):
    nik = data["nik"]
    expected_path = os.path.join(DIR_FASE2, f"wajah_{i:03d}_{nik}.jpg")
    
    if os.path.exists(expected_path):
        try:
            result = DeepFace.represent(img_path=expected_path, model_name=EMBEDDING_MODEL, enforce_detection=False)
            accepted_embeddings.append(result[0]["embedding"])
            data["wajah_path"] = expected_path
        except Exception:
            os.remove(expected_path)
            if "wajah_path" in data: del data["wajah_path"]
    else:
        if "wajah_path" in data: del data["wajah_path"]

print(f"✅ Sinkronisasi Selesai! Ada {len(accepted_embeddings)} wajah yang sudah aman di folder.")

# ==========================================
# 4. EKSEKUSI GENERATE DENGAN SDXL
# ==========================================
start_time = time.time()
print(f"\n🚀 Memulai Fase 2 (SDXL Engine)...")

for i, data in enumerate(final_datasets):
    if "wajah_path" in data: continue

    gender = data["jenis_kelamin"]
    nik = data["nik"]
    tgl_lahir = data.get("tgl_lahir", "1990")
    
    temp_path = os.path.join(DIR_FASE2, "temp_face.jpg")
    final_path = os.path.join(DIR_FASE2, f"wajah_{i:03d}_{nik}.jpg")

    print(f"\n[{i+1}/{len(final_datasets)}] Memproses Wajah untuk NIK {nik} ({gender})...")
    
    success = False
    attempts = 0

    while not success and attempts < 10:
        attempts += 1
        prompt, neg_prompt = get_sdxl_prompt(gender, tgl_lahir)

        # --- PERBAIKAN 1: PAKSA SDXL MEMAKAI SEED ACAK ---
        # Ini mencegah SDXL memulai render dari titik noise yang sama terus-menerus
        acak_seed = random.randint(0, 2147483647)
        gen = torch.Generator(device=DEVICE).manual_seed(acak_seed)

        # GENERATE MENGGUNAKAN SDXL
        image = pipe(
            prompt=prompt,
            negative_prompt=neg_prompt,
            num_inference_steps=55, # --- PERBAIKAN 2: STEP TURUN AGAR NGEBUT ---
            guidance_scale=7.5,
            width=768, 
            height=1024,
            generator=gen           # --- PERBAIKAN 3: SUNTIKKAN SEED ACAK KE SINI ---
        ).images[0]

        final_image = image.resize((300, 400))
        final_image.save(temp_path)

        # --- PERBAIKAN 4: HARDCODE THRESHOLD KE 0.15 ---
        # Angka 0.15 berarti: "Asal jarak wajahnya lebih dari 0.15, anggap itu orang yang berbeda!"
        is_unique, embedding, status_msg = verify_face_quality(temp_path, accepted_embeddings, 0.23)
        print(f"   - Attempt {attempts}: {status_msg}")

        if is_unique:
            os.rename(temp_path, final_path)
            accepted_embeddings.append(embedding)
            data["wajah_path"] = final_path
            print(f"   ✅ SUKSES! Wajah SDXL disimpan di: {final_path}")
            success = True
        else:
            if os.path.exists(temp_path): os.remove(temp_path)

    if not success:
        print(f"   ❌ GAGAL membuat wajah unik untuk data ke-{i+1} setelah 10 percobaan.")

    with open(FILE_FASE2, "w", encoding="utf-8") as f:
        json.dump(final_datasets, f, indent=4, ensure_ascii=False)

print(f"\n🎉 Fase 2 Selesai! Total wajah SDXL: {len(accepted_embeddings)}")


🔄 Melakukan Sinkronisasi Folder & Memori DeepFace...
✅ Sinkronisasi Selesai! Ada 99 wajah yang sudah aman di folder.

🚀 Memulai Fase 2 (SDXL Engine)...

[68/100] Memproses Wajah untuk NIK 7534986523094561 (PEREMPUAN)...


  0%|          | 0/55 [00:00<?, ?it/s]

   - Attempt 1: Ditolak: Mirip orang ke-65 (Jarak 0.09)


  0%|          | 0/55 [00:00<?, ?it/s]

   - Attempt 2: Ditolak: Mirip orang ke-66 (Jarak 0.22)


...

   - Attempt 5: Ditolak: Mirip orang ke-70 (Jarak 0.21)


  0%|          | 0/55 [00:00<?, ?it/s]

   - Attempt 6: Ditolak: Mirip orang ke-65 (Jarak 0.09)


  0%|          | 0/55 [00:00<?, ?it/s]

   - Attempt 7: Valid
   ✅ SUKSES! Wajah SDXL disimpan di: Fase2-Output\wajah_067_7534986523094561.jpg

🎉 Fase 2 Selesai! Total wajah SDXL: 100


# Fase 3 - Applying Output 1 & 2 to Master Template

In [ ]:
# from PIL import Image, ImageDraw, ImageFont
# import os

# # ==========================================
# # PREVIEW TESTER FASE 3.1 (MASTER GENERATOR)
# # ==========================================
# # Pastikan nama template dan foto wajah ini sesuai dengan yang ada di foldermu
# TEMPLATE_PATH = "Master-Template\Master-ktp.png" 
# DUMMY_PHOTO_PATH = "Fase2-Output\wajah_000_0123456789101112.jpg"
# OUTPUT_DIR = "Fase3.1-Output"
# OUTPUT_FILE = os.path.join(OUTPUT_DIR, "test_master_ktp.png")

# # Buat folder jika belum ada
# if not os.path.exists(OUTPUT_DIR):
#     os.makedirs(OUTPUT_DIR)

# def generate_test_master():
#     # 1. DATA FIKTIF UNTUK TESTING
#     data = {
#         "provinsi": "JAWA BARAT",
#         "kota": "KABUPATEN CIANJUR",
#         "nik": "3203012503770011",
#         "nama": "GUOHUI CHEN",
#         "ttl": "FUJIAN, 25-03-1977",
#         "jenis_kelamin": "MALE",
#         "golongan_darah": "-",
#         "alamat": "JL SELAMET PERUMAHAN RANCABALI",
#         "rt_rw": "002/004",
#         "kel_desa": "MUKA",
#         "kecamatan": "CIANJUR",
#         "agama": "CHRISTIAN",
#         "status": "MARRIED",
#         "pekerjaan": "OTHERS",
#         "kewarganegaraan": "CHINA",
#         "masa_berlaku": "12-12-2023",
#         "terbuat": "17-09-2018",
#     }
#     sign_name = data["nama"].split()[0]

#     # 2. BUKA TEMPLATE
#     if not os.path.exists(TEMPLATE_PATH):
#         return print(f"❌ Error: Template '{TEMPLATE_PATH}' tidak ditemukan!")
        
#     # Pastikan dalam mode RGBA untuk menunjang alpha_composite
#     tmp = Image.open(TEMPLATE_PATH).convert("RGBA")

#     # 3. LOAD FONT (Dengan Fallback Default)
#     try:
#         fprov = ImageFont.truetype("font/Arrial.ttf", 22)
#         fdata = ImageFont.truetype("font/Arrial.ttf", 16)
#         falamat = ImageFont.truetype("font/Arrial.ttf", 14)
        
#         # REVISI: Kecilkan ukuran font TTD dari 40 menjadi 20 atau 24
#         fsign = ImageFont.truetype("font/Sign.ttf", 20) 
        
#         fnik = ImageFont.truetype("font/Ocr.ttf", 26)
#         fnik_4x = ImageFont.truetype("font/Ocr.ttf", 26 * 4) 
#     except IOError:
#         print("⚠️ Peringatan: Font kustom tidak ditemukan!")
#         fprov = fdata = falamat = fsign = fnik = fnik_4x = ImageFont.load_default()

#     # 4. PASTE FOTO 
#     # REVISI: Tambahkan debug path untuk memastikan foto terbaca!
#     print(f"Mencari foto di: {os.path.abspath(DUMMY_PHOTO_PATH)}")
    
#     if os.path.exists(DUMMY_PHOTO_PATH):
#         pas_photo = Image.open(DUMMY_PHOTO_PATH)
#         target_w, target_h = 155, 199
#         pas_photo_final = pas_photo.resize((target_w, target_h))
        
#         # Tempelkan ke koordinat yang sudah kamu cari
#         tmp.paste(pas_photo_final, (444, 93))
#         print("✅ Foto berhasil ditempel!")
#     else:
#         print(f"❌ GAGAL: Foto '{DUMMY_PHOTO_PATH}' tidak ditemukan. Cek kembali lokasinya!")

#     # 5. SETUP WARNA DAN DRAW
#     warna_teks = (62, 62, 62, 255)
    
#     # ----------------------------------------------------
#     # KHUSUS NIK: Trik Layer 4x Scale untuk Ketebalan 0.4px
#     # ----------------------------------------------------
#     scale_factor = 4
#     nik_stroke_width = 0.4
    
#     # Buat layer transparan raksasa (4x lipat dari KTP)
#     nik_layer = Image.new("RGBA", (tmp.width * scale_factor, tmp.height * scale_factor), (0, 0, 0, 0))
#     nik_draw = ImageDraw.Draw(nik_layer)
    
#     # Gambar NIK di layer raksasa (Koordinat X,Y, Stroke juga dikali 4)
#     # Stroke fill di-set hitam pekat sesuai instruksimu
#     nik_draw.text(
#         (156 * scale_factor, 92 * scale_factor), 
#         data["nik"], 
#         fill=warna_teks, 
#         font=fnik_4x, 
#         anchor="lt", 
#         stroke_width=int(round(nik_stroke_width * scale_factor)), # 0.4 * 4 = 1.6 (Dibulatkan jadi 2px)
#         stroke_fill=(0, 0, 0, 255)
#     )
    
#     # Kecilkan kembali layer raksasa menggunakan LANCZOS (Anti-Aliasing kualitas tertinggi)
#     nik_layer_small = nik_layer.resize(tmp.size, Image.LANCZOS)
    
#     # Gabungkan layer NIK ke template KTP utama
#     tmp = Image.alpha_composite(tmp, nik_layer_small)
    
#     # ----------------------------------------------------
#     # RENDER TEKS LAINNYA
#     # ----------------------------------------------------
#     write = ImageDraw.Draw(tmp)
    
#     # Header
#     write.text((312,30), f"PROVINSI {data['provinsi']}", fill=warna_teks, font=fprov, anchor="ms")
#     write.text((311,36), data['kota'], fill=warna_teks, font=fprov, anchor="mt")
    
#     # Data Diri
#     write.text((170,124), data["nama"].upper(), fill=warna_teks, font=fdata, anchor="lt")
#     write.text((170,142), f"{data['ttl'].upper()}", fill=warna_teks, font=fdata, anchor="lt")
#     write.text((170,160), data["jenis_kelamin"].upper(), fill=warna_teks, font=fdata, anchor="lt")
#     write.text((313,160), f"Gol. Darah : {data['golongan_darah']}", fill=warna_teks, font=fdata, anchor="lt")
#     write.text((170,179), data["alamat"].upper(), fill=warna_teks, font=falamat, anchor="lt") # Ukuran 14
#     write.text((170,197), data["rt_rw"], fill=warna_teks, font=fdata, anchor="lt")
#     write.text((170,216), data["kel_desa"].upper(), fill=warna_teks, font=fdata, anchor="lt")
#     write.text((170,234), data["kecamatan"].upper(), fill=warna_teks, font=fdata, anchor="lt")
#     write.text((170,252), data["agama"].upper(), fill=warna_teks, font=fdata, anchor="lt")
#     write.text((170,270), data["status"].upper(), fill=warna_teks, font=fdata, anchor="lt")
#     write.text((170,289), data["pekerjaan"].upper(), fill=warna_teks, font=fdata, anchor="lt")
#     write.text((170,307), data["kewarganegaraan"].upper(), fill=warna_teks, font=fdata, anchor="lt")
#     write.text((170,325), data["masa_berlaku"].upper(), fill=warna_teks, font=fdata, anchor="lt")
    
#     # Footer
#     # Menghapus kata KABUPATEN/KOTA agar terlihat seperti KTP asli
#     kota_footer = data['kota'].replace("KOTA ", "").replace("KABUPATEN ", "")
#     write.text((525,299), kota_footer.upper(), fill=warna_teks, font=fdata, anchor="mt")
#     write.text((525,317), data["terbuat"], fill=warna_teks, font=fdata, anchor="mt")
#     write.text((525,341), sign_name, fill=warna_teks, font=fsign, anchor="mt")

#     # 6. SIMPAN HASIL
#     # Convert balik ke RGB karena format .jpg/.png standar KTP tidak butuh Alpha channel lagi
#     final_image = tmp.convert("RGB")
#     final_image.save(OUTPUT_FILE)
#     print(f"\n🎉 UJI COBA BERHASIL!")
#     print(f"Buka file ini untuk mengecek hasilnya: {OUTPUT_FILE}")

# if __name__ == "__main__":
#     generate_test_master()

Mencari foto di: o:\Fake-EKTP-Generator-With-Local-LLM-GAN-For-Data-Generator\Fase2-Output\wajah_000_0123456789101112.jpg
✅ Foto berhasil ditempel!

🎉 UJI COBA BERHASIL!
Buka file ini untuk mengecek hasilnya: Fase3.1-Output\test_master_ktp.png


In [9]:
import os
import json
import time
from PIL import Image, ImageDraw, ImageFont

# ==========================================
# FASE 3: AUTOMATED MASTER GENERATOR (2D BATCH RENDER)
# ==========================================
# Mendefinisikan ulang direktori agar aman dari Kernel Restart
DIR_FASE2 = "Fase2-Output"
FILE_FASE2 = os.path.join(DIR_FASE2, "data_fase2.json")
DIR_FASE3 = "Fase3-Output"
TEMPLATE_PATH = r"Master-Template\Master-ktp.png" 

# Buat folder Fase 3 jika belum ada
if not os.path.exists(DIR_FASE3):
    os.makedirs(DIR_FASE3)

FILE_FASE3 = os.path.join(DIR_FASE3, "data_fase3.json")

print("🔄 Memuat data dari Fase 2...")
if not os.path.exists(FILE_FASE2):
    raise FileNotFoundError(f"❌ File {FILE_FASE2} tidak ditemukan. Pastikan Fase 2 sudah selesai.")

with open(FILE_FASE2, "r", encoding="utf-8") as f:
    final_datasets = json.load(f)

# Memastikan template master ada
if not os.path.exists(TEMPLATE_PATH):
    raise FileNotFoundError(f"❌ Template Master '{TEMPLATE_PATH}' tidak ditemukan!")

# ==========================================
# LOAD FONT KE MEMORI (Dilakukan sekali agar cepat)
# ==========================================
try:
    fprov = ImageFont.truetype("font/Arrial.ttf", 22)
    fdata = ImageFont.truetype("font/Arrial.ttf", 16)
    falamat = ImageFont.truetype("font/Arrial.ttf", 14) # Khusus alamat agar muat
    fsign = ImageFont.truetype("font/Sign.ttf", 20) 
    
    # Font NIK 4x lipat untuk trik Anti-Aliasing Semi-Bold (Stroke 0.4px)
    fnik_4x = ImageFont.truetype("font/Ocr.ttf", 26 * 4) 
except IOError:
    print("⚠️ Peringatan: Font kustom tidak ditemukan! Pastikan folder 'font' ada.")
    fprov = fdata = falamat = fsign = fnik_4x = ImageFont.load_default()

# ==========================================
# EKSEKUSI BATCH RENDER 100 DATA
# ==========================================
print(f"🚀 Memulai Fase 3: Merender {len(final_datasets)} KTP Master Datar...")
start_time = time.time()
berhasil = 0

warna_teks = (62, 62, 62, 255)
scale_factor = 4
nik_stroke_width = 0.4

for i, data in enumerate(final_datasets):
    try:
        # 1. Buka Template Baru untuk setiap data
        tmp = Image.open(TEMPLATE_PATH).convert("RGBA")
        
        # 2. Paste Pas Foto SDXL
        photo_path = data.get("wajah_path", "")
        if photo_path and os.path.exists(photo_path):
            pas_photo = Image.open(photo_path)
            target_w, target_h = 155, 199
            pas_photo_final = pas_photo.resize((target_w, target_h))
            tmp.paste(pas_photo_final.convert("RGBA"), (444, 93))
        else:
            print(f"   ⚠️ Data ke-{i+1}: Foto tidak ditemukan di '{photo_path}', mengosongkan area foto.")

        # 3. Layer Khusus NIK (Anti-Aliasing Semi-Bold 0.4px)
        nik_layer = Image.new("RGBA", (tmp.width * scale_factor, tmp.height * scale_factor), (0, 0, 0, 0))
        nik_draw = ImageDraw.Draw(nik_layer)
        
        nik_draw.text(
            (156 * scale_factor, 92 * scale_factor), 
            data.get("nik", ""), 
            fill=warna_teks, 
            font=fnik_4x, 
            anchor="lt", 
            stroke_width=int(round(nik_stroke_width * scale_factor)), # Menjadi 1.6px -> 2px di layer raksasa
            stroke_fill=(0, 0, 0, 255)
        )
        
        # Resize Lanczos & Gabungkan Layer NIK
        nik_layer_small = nik_layer.resize(tmp.size, Image.LANCZOS)
        tmp = Image.alpha_composite(tmp, nik_layer_small)
        
        # 4. Render Teks Utama
        write = ImageDraw.Draw(tmp)
        
        # Header
        write.text((312, 30), f"PROVINSI {data.get('provinsi', '')}".upper(), fill=warna_teks, font=fprov, anchor="ms")
        write.text((311, 36), data.get('kota_kab', '').upper(), fill=warna_teks, font=fprov, anchor="mt")
        
        # Data Diri Utama
        write.text((170, 124), data.get("nama", "").upper(), fill=warna_teks, font=fdata, anchor="lt")
        ttl_str = f"{data.get('tempat_lahir', '')}, {data.get('tgl_lahir', '')}"
        write.text((170, 142), ttl_str.upper(), fill=warna_teks, font=fdata, anchor="lt")
        write.text((170, 160), data.get("jenis_kelamin", "").upper(), fill=warna_teks, font=fdata, anchor="lt")
        write.text((313, 160), f"Gol. Darah : {data.get('gol_darah', '').upper()}", fill=warna_teks, font=fdata, anchor="lt")
        write.text((170, 179), data.get("alamat", "").upper(), fill=warna_teks, font=falamat, anchor="lt") # Font 14
        write.text((170, 197), data.get("rt_rw", ""), fill=warna_teks, font=fdata, anchor="lt")
        write.text((170, 216), data.get("kel_desa", "").upper(), fill=warna_teks, font=fdata, anchor="lt")
        write.text((170, 234), data.get("kecamatan", "").upper(), fill=warna_teks, font=fdata, anchor="lt")
        write.text((170, 252), data.get("agama", "").upper(), fill=warna_teks, font=fdata, anchor="lt")
        write.text((170, 270), data.get("status_perkawinan", "").upper(), fill=warna_teks, font=fdata, anchor="lt")
        write.text((170, 289), data.get("pekerjaan", "").upper(), fill=warna_teks, font=fdata, anchor="lt")
        write.text((170, 307), data.get("kewarganegaraan", "").upper(), fill=warna_teks, font=fdata, anchor="lt")
        write.text((170, 325), data.get("berlaku_hingga", "").upper(), fill=warna_teks, font=fdata, anchor="lt")
        
        # Footer
        kota_footer = data.get('kota_kab', '').replace("KOTA ", "").replace("KABUPATEN ", "")
        write.text((525, 299), kota_footer.upper(), fill=warna_teks, font=fdata, anchor="mt")
        write.text((525, 317), data.get("tgl_pembuatan", ""), fill=warna_teks, font=fdata, anchor="mt")
        
        # Tanda Tangan
        sign_name = data.get("nama", "A").split()[0]
        write.text((525, 341), sign_name, fill=warna_teks, font=fsign, anchor="mt")

        # 5. Simpan Hasil ke Fase3-Output
        final_image = tmp.convert("RGB")
        nik_file = data.get("nik", f"UNKNOWN_{i}")
        output_filename = f"master_ktp_{nik_file}.png"
        output_path = os.path.join(DIR_FASE3, output_filename)
        
        tmp.save(output_path, "PNG")
        
        # 6. Update Path di JSON untuk diteruskan ke Fase 4
        data["master_ktp_path"] = output_path
        berhasil += 1
        
        if (i + 1) % 10 == 0:
            print(f"   [+] Merender {i + 1}/100 Master KTP selesai...")
            
    except Exception as e:
        print(f"❌ Error pada dataset ke-{i} (NIK: {data.get('nik')}): {e}")

# ==========================================
# PENYIMPANAN JSON ESTAFET (FASE 3)
# ==========================================
with open(FILE_FASE3, "w", encoding="utf-8") as f:
    json.dump(final_datasets, f, indent=4, ensure_ascii=False)

end_time = time.time()
print("="*50)
print(f"✅ FASE 3 SELESAI ({berhasil}/{len(final_datasets)} berhasil dirender)")
print(f"Waktu Eksekusi: {end_time - start_time:.2f} detik.")
print(f"Kumpulan Gambar Master KTP tersimpan di : {DIR_FASE3}")
print(f"Data Estafet tersimpan di file          : {FILE_FASE3}")
print("="*50)

🔄 Memuat data dari Fase 2...
🚀 Memulai Fase 3: Merender 100 KTP Master Datar...
   [+] Merender 10/100 Master KTP selesai...
   [+] Merender 20/100 Master KTP selesai...
   [+] Merender 30/100 Master KTP selesai...
   [+] Merender 40/100 Master KTP selesai...
   [+] Merender 50/100 Master KTP selesai...
   [+] Merender 60/100 Master KTP selesai...
   [+] Merender 70/100 Master KTP selesai...
   [+] Merender 80/100 Master KTP selesai...
   [+] Merender 90/100 Master KTP selesai...
   [+] Merender 100/100 Master KTP selesai...
✅ FASE 3 SELESAI (100/100 berhasil dirender)
Waktu Eksekusi: 11.80 detik.
Kumpulan Gambar Master KTP tersimpan di : Fase3-Output
Data Estafet tersimpan di file          : Fase3-Output\data_fase3.json


# Fase 3.1 - Geometrical Caliration & Polygon Mask

In [10]:
import cv2
import os
import json
import numpy as np

# ==========================================
# ALAT KALIBRASI TEMPLATE CLEAN (FASE 3.1)
# ==========================================
TEMPLATE_DIR = "Template-Clean"
CONFIG_DIR = "Fase3.1-Output"
CONFIG_FILE = os.path.join(CONFIG_DIR, "config_clean_templates.json")
PADDING = 400

if not os.path.exists(CONFIG_DIR):
    os.makedirs(CONFIG_DIR)

# Hanya 2 field yang dibutuhkan: 4 Sudut KTP & Poligon Tangan
REQUIRED_FIELDS = [
    ("ktp_corners", "4 Sudut KTP (Luar/Ujung Kartu)", 4, (0, 255, 0)),
    ("hand_mask", "Poligon Jari/Tangan (ENTER=Selesai, S=Tidak Ada/Skip)", -1, (0, 0, 255))
]

template_configs = []
current_entry = {}
current_field_idx = 0
temp_display = []
temp_original = []
current_scale = 1.0

def mouse_click(event, x, y, flags, param):
    global temp_display, temp_original, current_scale, current_field_idx
    if event == cv2.EVENT_LBUTTONDOWN and current_field_idx < len(REQUIRED_FIELDS):
        key, label, req_count, color = REQUIRED_FIELDS[current_field_idx]
        # Jika count = -1 (Poligon bebas) atau titik belum mencapai batas
        if req_count == -1 or len(temp_display) < req_count:
            temp_display.append([x, y])
            
            # Konversi kordinat layar ke kordinat asli dikurangi Padding
            orig_x = int(x / current_scale) - PADDING
            orig_y = int(y / current_scale) - PADDING
            temp_original.append([orig_x, orig_y])
            print(f"[{label}] Titik ke-{len(temp_display)}: Layar({x},{y}) -> Asli({orig_x},{orig_y})")

def draw_saved_data(img, entry, scale):
    # Gambar 4 Sudut KTP (Warna Hijau)
    if "ktp_corners" in entry and entry["ktp_corners"]:
        pts = [(int((p[0]+PADDING)*scale), int((p[1]+PADDING)*scale)) for p in entry["ktp_corners"]]
        for i, p in enumerate(pts):
            cv2.circle(img, p, 4, (0,255,0), -1)
            if i > 0: cv2.line(img, pts[i-1], p, (0,255,0), 2)
        if len(pts) == 4: cv2.line(img, pts[3], pts[0], (0,255,0), 2)
    
    # Gambar Poligon Tangan (Warna Merah)
    if "hand_mask" in entry and entry["hand_mask"]:
        pts = [(int((p[0]+PADDING)*scale), int((p[1]+PADDING)*scale)) for p in entry["hand_mask"]]
        for i, p in enumerate(pts):
            cv2.circle(img, p, 4, (0,0,255), -1)
            if i > 0: cv2.line(img, pts[i-1], p, (0,0,255), 2)
        if len(pts) > 2: cv2.line(img, pts[-1], pts[0], (0,0,255), 2)

def run_calibrator():
    global template_configs, current_entry, current_field_idx, temp_display, temp_original, current_scale
    
    if not os.path.exists(TEMPLATE_DIR):
        print(f"❌ Folder '{TEMPLATE_DIR}' tidak ditemukan! Silakan buat dan isi dengan 9 template clean.")
        return

    template_files = sorted([f for f in os.listdir(TEMPLATE_DIR) if f.endswith(('.jpg', '.png', '.jpeg'))])
    if not template_files:
        print(f"❌ Folder '{TEMPLATE_DIR}' kosong.")
        return

    if os.path.exists(CONFIG_FILE):
        try:
            with open(CONFIG_FILE, "r") as f: template_configs = json.load(f)
            print(f"🔄 Berhasil memuat {len(template_configs)} data kalibrasi lama.")
        except: template_configs = []

    print("\n" + "="*50)
    print("=== KALIBRASI GEOMETRI TEMPLATE CLEAN (FASE 3.1) ===")
    print("="*50)
    print(f"⚠️ Disediakan ruang {PADDING}px di sekeliling gambar.")
    print("KONTROL:")
    print(" - Klik Kiri      : Tambah titik")
    print(" - Tombol [S]     : Skip / Tidak ada jari (Hanya di mode Tangan)")
    print(" - Tombol [ENTER] : Selesai membuat poligon tangan")
    print(" - Tombol [R]     : Reset titik yang sedang dibuat")
    print(" - Tombol [Q]     : Simpan & Keluar sementara")
    print("="*50)

    for filepath in template_files:
        filename = os.path.basename(filepath)
        full_path = os.path.join(TEMPLATE_DIR, filename)
        
        # Cari data gambar ini di konfigurasi
        current_entry = next((item for item in template_configs if item.get("template_name") == filename), None)
        if not current_entry:
            current_entry = {
                "template_name": filename,
                "template_path": full_path,
                "mask_path": os.path.join(CONFIG_DIR, f"mask_{filename.split('.')[0]}.png")
            }
            template_configs.append(current_entry)

        # Cek apakah kalibrasi gambar ini sudah selesai
        current_field_idx = 0
        is_complete = True
        for i, field in enumerate(REQUIRED_FIELDS):
            key = field[0]
            if key not in current_entry:
                current_field_idx = i
                is_complete = False
                break
        
        if is_complete:
            print(f"⏩ SKIP: '{filename}' sudah dikalibrasi.")
            continue

        img = cv2.imread(full_path)
        if img is None: continue
        
        # Menambahkan virtual padding abu-abu
        padded_img = cv2.copyMakeBorder(img, PADDING, PADDING, PADDING, PADDING, cv2.BORDER_CONSTANT, value=[40, 40, 40])
        h, w = padded_img.shape[:2]
        
        # Scale gambar agar pas di layar monitor
        max_dim = 900 
        current_scale = 1.0 if max(h, w) <= max_dim else max_dim / float(max(h, w))
        display_base = cv2.resize(padded_img, (int(w * current_scale), int(h * current_scale)))
        
        temp_display, temp_original = [], []
        window_name = f"Kalibrasi: {filename}"
        cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE) 
        cv2.setMouseCallback(window_name, mouse_click)

        while True:
            display_img = display_base.copy()
            draw_saved_data(display_img, current_entry, current_scale)
            
            if current_field_idx < len(REQUIRED_FIELDS):
                key, label, req_count, color = REQUIRED_FIELDS[current_field_idx]
                
                # Panel Instruksi di Layar
                cv2.rectangle(display_img, (10, 10), (800, 70), (0, 0, 0), -1)
                cv2.putText(display_img, f"SEKARANG: {label}", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
                
                # Menggambar titik yang sedang diklik secara Live
                for i, p in enumerate(temp_display):
                    cv2.circle(display_img, p, 4, color, -1)
                    if i > 0: cv2.line(display_img, temp_display[i-1], p, color, 2)
                    # Menutup garis otomatis jika 4 titik KTP
                    if req_count == 4 and len(temp_display) == 4 and i == 3: 
                        cv2.line(display_img, temp_display[3], temp_display[0], color, 2)
                    # Menutup garis putus-putus bantuan untuk mode Poligon Bebas
                    elif req_count == -1 and len(temp_display) > 2 and i == len(temp_display)-1:
                        cv2.line(display_img, temp_display[-1], temp_display[0], (150,150,150), 1)
                
                # JIKA MODE 4 TITIK KTP SELESAI
                if req_count == 4 and len(temp_display) == 4:
                    current_entry[key] = temp_original
                    print(f"💾 Kordinat '{key}' tersimpan!")
                    current_field_idx += 1
                    temp_display, temp_original = [], []
                    continue

            cv2.imshow(window_name, display_img)
            key_code = cv2.waitKey(20) & 0xFF
            
            if key_code in [ord('r'), ord('R')]:
                temp_display, temp_original = [], []
                print("🔄 Reset field aktif.")
            
            elif key_code in [ord('q'), ord('Q')]:
                with open(CONFIG_FILE, "w") as f: json.dump(template_configs, f, indent=4)
                cv2.destroyAllWindows()
                return
                
            # JIKA MODE TANGAN, DAN DITEKAN 'S' (TIDAK ADA TANGAN)
            elif key_code in [ord('s'), ord('S')] and current_field_idx == 1:
                current_entry["hand_mask"] = []
                print("⏩ Skip Poligon Tangan (Tidak ada jari di template ini).")
                
                # GENERATE MASK SOLID (Karena tidak ada jari)
                ktp_pts = np.array(current_entry["ktp_corners"], dtype=np.int32)
                mask = np.zeros(img.shape[:2], dtype=np.uint8)
                cv2.fillPoly(mask, [ktp_pts], 255)
                cv2.imwrite(current_entry["mask_path"], mask)
                
                with open(CONFIG_FILE, "w") as f: json.dump(template_configs, f, indent=4)
                current_field_idx += 1
                break # Pindah ke gambar selanjutnya
                
            # JIKA MODE TANGAN, DAN DITEKAN 'ENTER' (SELESAI MENGGAMBAR POLIGON)
            elif key_code == 13 and current_field_idx == 1: 
                if len(temp_display) >= 3 or len(temp_display) == 0:
                    current_entry["hand_mask"] = temp_original
                    
                    # GENERATE MASK BOLONG (KTP Putih, Tangan Hitam)
                    ktp_pts = np.array(current_entry["ktp_corners"], dtype=np.int32)
                    mask = np.zeros(img.shape[:2], dtype=np.uint8)
                    cv2.fillPoly(mask, [ktp_pts], 255) # Warnai area KTP jadi putih
                    
                    if len(temp_original) >= 3:
                        hand_pts = np.array(temp_original, dtype=np.int32)
                        cv2.fillPoly(mask, [hand_pts], 0) # Warnai area jari jadi hitam (Bolong)
                        
                    cv2.imwrite(current_entry["mask_path"], mask)
                    
                    with open(CONFIG_FILE, "w") as f: json.dump(template_configs, f, indent=4)
                    print(f"🖼️ Mask Poligon tersimpan mantap!")
                    current_field_idx += 1
                    break # Pindah ke gambar selanjutnya
                else:
                    print("⚠️ Minimal butuh 3 titik untuk membuat poligon tangan! (Atau tekan 'S' jika tidak ada jari)")

        cv2.destroyWindow(window_name)

    print(f"\n✅ SEMUA 9 TEMPLATE CLEAN BERES!")
    print(f"File Konfigurasi Geometri tersimpan di : {CONFIG_FILE}")
    print(f"Gambar Mask Transparansi tersimpan di  : {CONFIG_DIR}")

if __name__ == "__main__":
    run_calibrator()


=== KALIBRASI GEOMETRI TEMPLATE CLEAN (FASE 3.1) ===
⚠️ Disediakan ruang 400px di sekeliling gambar.
KONTROL:
 - Klik Kiri      : Tambah titik
 - Tombol [S]     : Skip / Tidak ada jari (Hanya di mode Tangan)
 - Tombol [ENTER] : Selesai membuat poligon tangan
 - Tombol [R]     : Reset titik yang sedang dibuat
 - Tombol [Q]     : Simpan & Keluar sementara
[4 Sudut KTP (Luar/Ujung Kartu)] Titik ke-1: Layar(313,260) -> Asli(208,105)
[4 Sudut KTP (Luar/Ujung Kartu)] Titik ke-2: Layar(641,251) -> Asli(846,88)
[4 Sudut KTP (Luar/Ujung Kartu)] Titik ke-3: Layar(648,461) -> Asli(860,496)
[4 Sudut KTP (Luar/Ujung Kartu)] Titik ke-4: Layar(312,467) -> Asli(206,508)
💾 Kordinat 'ktp_corners' tersimpan!
[Poligon Jari/Tangan (ENTER=Selesai, S=Tidak Ada/Skip)] Titik ke-1: Layar(204,310) -> Asli(-4,202)
[Poligon Jari/Tangan (ENTER=Selesai, S=Tidak Ada/Skip)] Titik ke-2: Layar(276,205) -> Asli(136,-2)
[Poligon Jari/Tangan (ENTER=Selesai, S=Tidak Ada/Skip)] Titik ke-3: Layar(421,204) -> Asli(418,-4)
[Po

In [3]:
import cv2
import os
import json
import numpy as np

# ==========================================
# PREVIEW TESTER MASKING & GEOMETRI (FASE 3.1)
# ==========================================
CONFIG_DIR = "Fase3.1-Output"
CONFIG_FILE = os.path.join(CONFIG_DIR, "config_clean_templates.json")
PREVIEW_DIR = "Preview-Masking"

# Buat folder preview jika belum ada
if not os.path.exists(PREVIEW_DIR):
    os.makedirs(PREVIEW_DIR)

def generate_mask_preview():
    if not os.path.exists(CONFIG_FILE):
        return print(f"❌ Error: File {CONFIG_FILE} tidak ditemukan. Pastikan Fase 3.1 sudah selesai.")
        
    with open(CONFIG_FILE, "r") as f:
        template_configs = json.load(f)
        
    print(f"🔍 Membuat preview masking untuk {len(template_configs)} template...")
    
    for entry in template_configs:
        img_path = entry.get("template_path")
        filename = entry.get("template_name", "unknown.jpg")
        
        if not os.path.exists(img_path):
            print(f"   ⚠️ Gambar tidak ditemukan: {img_path}")
            continue
            
        # Baca gambar asli
        img = cv2.imread(img_path)
        overlay = img.copy()
        
        # 1. Gambar Area KTP (HIJAU Tembus Pandang)
        if "ktp_corners" in entry and entry["ktp_corners"]:
            ktp_pts = np.array(entry["ktp_corners"], dtype=np.int32)
            cv2.fillPoly(overlay, [ktp_pts], (0, 255, 0))
            
        # 2. Gambar Area Tangan/Jari (MERAH Tembus Pandang)
        # REVISI: Mendukung Multi-Poligon (Banyak Jari)
        if "hand_mask" in entry and entry["hand_mask"]:
            saved_masks = entry["hand_mask"]
            
            # Cek apakah datanya format lama (1 jari) atau baru (multi-jari)
            if len(saved_masks) > 0 and isinstance(saved_masks[0][0], int):
                # Format lama (satu list koordinat datar)
                hand_pts = np.array(saved_masks, dtype=np.int32)
                cv2.fillPoly(overlay, [hand_pts], (0, 0, 255))
            else:
                # Format baru (list of list koordinat)
                for poly in saved_masks:
                    if len(poly) >= 3: # Pastikan poligon valid (minimal 3 titik)
                        hand_pts = np.array(poly, dtype=np.int32)
                        cv2.fillPoly(overlay, [hand_pts], (0, 0, 255))
            
        # 3. Gabungkan overlay dengan gambar asli (Opacity 50%)
        alpha = 0.5  # Transparansi 50%
        cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0, img)
        
        # 4. Tambahkan teks nama file di pojok kiri atas
        cv2.putText(img, filename, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
        
        # Simpan hasil preview
        out_path = os.path.join(PREVIEW_DIR, f"preview_{filename}")
        cv2.imwrite(out_path, img)
        print(f"   ✅ Preview tersimpan: {out_path}")
        
    print(f"\n🎉 Selesai! Silakan cek folder '{PREVIEW_DIR}' untuk melihat hasilnya.")

if __name__ == "__main__":
    generate_mask_preview()

🔍 Membuat preview masking untuk 10 template...
   ✅ Preview tersimpan: Preview-Masking\preview_ktp01.png
   ✅ Preview tersimpan: Preview-Masking\preview_ktp02.png
   ✅ Preview tersimpan: Preview-Masking\preview_ktp03.png
   ✅ Preview tersimpan: Preview-Masking\preview_ktp04.png
   ✅ Preview tersimpan: Preview-Masking\preview_ktp05.png
   ✅ Preview tersimpan: Preview-Masking\preview_ktp06.png
   ✅ Preview tersimpan: Preview-Masking\preview_ktp07.png
   ✅ Preview tersimpan: Preview-Masking\preview_ktp08.png
   ✅ Preview tersimpan: Preview-Masking\preview_ktp09.png
   ✅ Preview tersimpan: Preview-Masking\preview_ktp10.png

🎉 Selesai! Silakan cek folder 'Preview-Masking' untuk melihat hasilnya.


In [1]:
import cv2
import os
import json
import numpy as np

# ==========================================
# ALAT REVISI MULTI-POLYGON (FASE 3.1)
# ==========================================
TEMPLATE_DIR = "Template-Clean"
CONFIG_DIR = "Fase3.1-Output"
CONFIG_FILE = os.path.join(CONFIG_DIR, "config_clean_templates.json")
PADDING = 400

REQUIRED_FIELDS = [
    ("ktp_corners", "4 Sudut KTP", 4, (0, 255, 0)),
    ("hand_mask", "Poligon Jari (SPASI=Kunci 1 Jari, ENTER=Selesai Semua)", -1, (0, 0, 255))
]

template_configs = []
current_entry = {}
current_field_idx = 0

# Variabel untuk menampung titik yang sedang aktif (live)
temp_display = []
temp_original = []

# Variabel BARU untuk menampung poligon-poligon jari yang sudah di-SPASI
completed_fingers_display = []
completed_fingers_original = []

current_scale = 1.0

def mouse_click(event, x, y, flags, param):
    global temp_display, temp_original, current_scale, current_field_idx
    if event == cv2.EVENT_LBUTTONDOWN and current_field_idx < len(REQUIRED_FIELDS):
        key, label, req_count, color = REQUIRED_FIELDS[current_field_idx]
        if req_count == -1 or len(temp_display) < req_count:
            temp_display.append([x, y])
            orig_x = int(x / current_scale) - PADDING
            orig_y = int(y / current_scale) - PADDING
            temp_original.append([orig_x, orig_y])

def draw_saved_data(img, entry, scale):
    if "ktp_corners" in entry and entry["ktp_corners"]:
        pts = [(int((p[0]+PADDING)*scale), int((p[1]+PADDING)*scale)) for p in entry["ktp_corners"]]
        for i, p in enumerate(pts):
            cv2.circle(img, p, 4, (0,255,0), -1)
            if i > 0: cv2.line(img, pts[i-1], p, (0,255,0), 2)
        if len(pts) == 4: cv2.line(img, pts[3], pts[0], (0,255,0), 2)
    
    # Render masking lama (Support format lama 1 jari & format baru multi-jari)
    if "hand_mask" in entry and entry["hand_mask"]:
        saved_masks = entry["hand_mask"]
        # Konversi format lama ke list of lists jika perlu
        if len(saved_masks) > 0 and isinstance(saved_masks[0][0], int):
            saved_masks = [saved_masks]
            
        for poly in saved_masks:
            pts = [(int((p[0]+PADDING)*scale), int((p[1]+PADDING)*scale)) for p in poly]
            for i, p in enumerate(pts):
                cv2.circle(img, p, 4, (0,0,255), -1)
                if i > 0: cv2.line(img, pts[i-1], p, (0,0,255), 2)
            if len(pts) > 2: cv2.line(img, pts[-1], pts[0], (0,0,255), 2)

def run_reviser():
    global template_configs, current_entry, current_field_idx
    global temp_display, temp_original, completed_fingers_display, completed_fingers_original, current_scale
    
    if not os.path.exists(CONFIG_FILE):
        return print(f"❌ Error: File {CONFIG_FILE} tidak ditemukan!")

    with open(CONFIG_FILE, "r") as f:
        template_configs = json.load(f)

    while True:
        print("\n" + "="*50)
        print("=== MENU REVISI MULTI-POLYGON ===")
        print("="*50)
        for i, entry in enumerate(template_configs):
            print(f"[{i}] {entry['template_name']}")
        print("[Q] Keluar dan Simpan")
        print("="*50)
        
        choice = input("Masukkan NOMOR template yang ingin direvisi (atau Q): ").strip().upper()
        
        if choice == 'Q':
            print("💾 Keluar dari program revisi.")
            break
            
        if not choice.isdigit() or int(choice) < 0 or int(choice) >= len(template_configs):
            print("⚠️ Pilihan tidak valid.")
            continue
            
        idx = int(choice)
        current_entry = template_configs[idx]
        img_path = current_entry["template_path"]
        
        if not os.path.exists(img_path):
            continue

        print(f"\n🔄 MEMULAI REVISI: {current_entry['template_name']}")
        
        current_entry.pop("ktp_corners", None)
        current_entry.pop("hand_mask", None)
        current_field_idx = 0
        
        img = cv2.imread(img_path)
        padded_img = cv2.copyMakeBorder(img, PADDING, PADDING, PADDING, PADDING, cv2.BORDER_CONSTANT, value=[40, 40, 40])
        h, w = padded_img.shape[:2]
        
        max_dim = 900 
        current_scale = 1.0 if max(h, w) <= max_dim else max_dim / float(max(h, w))
        display_base = cv2.resize(padded_img, (int(w * current_scale), int(h * current_scale)))
        
        temp_display, temp_original = [], []
        completed_fingers_display, completed_fingers_original = [], []
        
        window_name = f"Revisi Multi-Jari: {current_entry['template_name']}"
        cv2.namedWindow(window_name, cv2.WINDOW_AUTOSIZE) 
        cv2.setMouseCallback(window_name, mouse_click)

        while True:
            display_img = display_base.copy()
            draw_saved_data(display_img, current_entry, current_scale)
            
            if current_field_idx < len(REQUIRED_FIELDS):
                key, label, req_count, color = REQUIRED_FIELDS[current_field_idx]
                
                cv2.rectangle(display_img, (10, 10), (800, 70), (0, 0, 0), -1)
                cv2.putText(display_img, f"SEKARANG: {label}", (20, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
                
                # --- RENDER JARI YANG SUDAH DIKUNCI (SPASI) ---
                if current_field_idx == 1:
                    for poly in completed_fingers_display:
                        for i, p in enumerate(poly):
                            cv2.circle(display_img, tuple(p), 4, (0, 200, 255), -1) # Warna orange untuk yg sudah dikunci
                            if i > 0: cv2.line(display_img, tuple(poly[i-1]), tuple(p), (0, 200, 255), 2)
                        if len(poly) > 2: cv2.line(display_img, tuple(poly[-1]), tuple(poly[0]), (0, 200, 255), 2)
                
                # --- RENDER JARI YANG SEDANG DIBUAT ---
                for i, p in enumerate(temp_display):
                    cv2.circle(display_img, tuple(p), 4, color, -1)
                    if i > 0: cv2.line(display_img, tuple(temp_display[i-1]), tuple(p), color, 2)
                    if req_count == 4 and len(temp_display) == 4 and i == 3: 
                        cv2.line(display_img, tuple(temp_display[3]), tuple(temp_display[0]), color, 2)
                    elif req_count == -1 and len(temp_display) > 2 and i == len(temp_display)-1:
                        cv2.line(display_img, tuple(temp_display[-1]), tuple(temp_display[0]), (150,150,150), 1)
                
                if req_count == 4 and len(temp_display) == 4:
                    current_entry[key] = temp_original
                    current_field_idx += 1
                    temp_display, temp_original = [], []
                    continue

            cv2.imshow(window_name, display_img)
            key_code = cv2.waitKey(20) & 0xFF
            
            if key_code in [ord('r'), ord('R')]:
                temp_display, temp_original = [], []
                completed_fingers_display, completed_fingers_original = [], [] # Reset semua jari
                print("🔄 Reset field aktif.")
            
            elif key_code in [ord('q'), ord('Q')]:
                break
                
            elif key_code in [ord('s'), ord('S')] and current_field_idx == 1:
                # Mode SKIP jari (Tidak ada jari sama sekali)
                current_entry["hand_mask"] = []
                ktp_pts = np.array(current_entry["ktp_corners"], dtype=np.int32)
                mask = np.zeros(img.shape[:2], dtype=np.uint8)
                cv2.fillPoly(mask, [ktp_pts], 255)
                cv2.imwrite(current_entry["mask_path"], mask)
                with open(CONFIG_FILE, "w") as f: json.dump(template_configs, f, indent=4)
                print("✅ Revisi Selesai (Tidak ada jari).")
                break
                
            elif key_code == 32 and current_field_idx == 1: 
                # TOMBOL [SPASI]: KUNCI 1 JARI
                if len(temp_original) >= 3:
                    completed_fingers_display.append(temp_display)
                    completed_fingers_original.append(temp_original)
                    print(f"🔒 Terkunci! Jari ke-{len(completed_fingers_original)} disimimpan. Silakan klik area jari lain.")
                    temp_display, temp_original = [], []
                else:
                    print("⚠️ Butuh minimal 3 titik untuk mengunci jari ini!")
                    
            elif key_code == 13 and current_field_idx == 1: 
                # TOMBOL [ENTER]: FINALISASI SEMUA JARI
                # Jika ada titik menggantung yang belum di-spasi, masukkan juga
                if len(temp_original) >= 3:
                    completed_fingers_original.append(temp_original)
                
                if len(completed_fingers_original) > 0:
                    current_entry["hand_mask"] = completed_fingers_original
                    
                    ktp_pts = np.array(current_entry["ktp_corners"], dtype=np.int32)
                    mask = np.zeros(img.shape[:2], dtype=np.uint8)
                    cv2.fillPoly(mask, [ktp_pts], 255) # Tembak Hijau/Putih KTP
                    
                    # Bolongi KTP dengan semua jari yang dikumpulkan
                    for poly in completed_fingers_original:
                        hand_pts = np.array(poly, dtype=np.int32)
                        cv2.fillPoly(mask, [hand_pts], 0) # Tembak Merah/Hitam Jari
                        
                    cv2.imwrite(current_entry["mask_path"], mask)
                    with open(CONFIG_FILE, "w") as f: json.dump(template_configs, f, indent=4)
                    print(f"✅ Revisi Selesai (Total {len(completed_fingers_original)} jari tersimpan).")
                    break
                elif len(completed_fingers_original) == 0 and len(temp_original) == 0:
                    print("⚠️ Kosong! Tekan 'S' jika memang tidak ada jari.")
                    
        cv2.destroyWindow(window_name)

if __name__ == "__main__":
    run_reviser()


=== MENU REVISI MULTI-POLYGON ===
[0] ktp01.png
[1] ktp02.png
[2] ktp03.png
[3] ktp04.png
[4] ktp05.png
[5] ktp06.png
[6] ktp07.png
[7] ktp08.png
[8] ktp09.png
[9] ktp10.png
[Q] Keluar dan Simpan

🔄 MEMULAI REVISI: ktp05.png
🔒 Terkunci! Jari ke-1 disimimpan. Silakan klik area jari lain.
🔒 Terkunci! Jari ke-2 disimimpan. Silakan klik area jari lain.
🔒 Terkunci! Jari ke-3 disimimpan. Silakan klik area jari lain.
✅ Revisi Selesai (Total 4 jari tersimpan).

=== MENU REVISI MULTI-POLYGON ===
[0] ktp01.png
[1] ktp02.png
[2] ktp03.png
[3] ktp04.png
[4] ktp05.png
[5] ktp06.png
[6] ktp07.png
[7] ktp08.png
[8] ktp09.png
[9] ktp10.png
[Q] Keluar dan Simpan

🔄 MEMULAI REVISI: ktp06.png
🔒 Terkunci! Jari ke-1 disimimpan. Silakan klik area jari lain.
🔒 Terkunci! Jari ke-2 disimimpan. Silakan klik area jari lain.
✅ Revisi Selesai (Total 3 jari tersimpan).

=== MENU REVISI MULTI-POLYGON ===
[0] ktp01.png
[1] ktp02.png
[2] ktp03.png
[3] ktp04.png
[4] ktp05.png
[5] ktp06.png
[6] ktp07.png
[7] ktp08.png


# Fase 3.2: Automated Batch Warping & Masking Execution

In [3]:
import os
import json
import cv2
import random
import time
import numpy as np

# ==========================================
# FASE 3.2: BATCH WARPING & ALPHA MASKING
# ==========================================
DIR_FASE3 = "Fase3-Output"
FILE_FASE3 = os.path.join(DIR_FASE3, "data_fase3.json")

DIR_FASE3_1 = "Fase3.1-Output"
CONFIG_TEMPLATE = os.path.join(DIR_FASE3_1, "config_clean_templates.json")

DIR_FASE3_2 = "Fase3.2-Output"
FILE_FASE3_2 = os.path.join(DIR_FASE3_2, "data_fase3_2.json")

if not os.path.exists(DIR_FASE3_2):
    os.makedirs(DIR_FASE3_2)

# 1. LOAD DATA
print("🔄 Memuat data rekonstruksi...")
if not os.path.exists(FILE_FASE3):
    raise FileNotFoundError(f"❌ File {FILE_FASE3} tidak ditemukan. Selesaikan Fase 3 terlebih dahulu.")
if not os.path.exists(CONFIG_TEMPLATE):
    raise FileNotFoundError(f"❌ File geometri {CONFIG_TEMPLATE} tidak ditemukan.")

with open(FILE_FASE3, "r", encoding="utf-8") as f:
    fase3_dataset = json.load(f)

with open(CONFIG_TEMPLATE, "r", encoding="utf-8") as f:
    template_configs = json.load(f)

print(f"✅ Terdeteksi {len(fase3_dataset)} dataset KTP Master transparan.")
print(f"✅ Terdeteksi {len(template_configs)} konfigurasi Template Clean siap pakai.")

# 2. PROSES BATCH PROCESSING
print(f"\n🚀 Memulai orkestrasi Homografi & Alpha Blending untuk {len(fase3_dataset)} data...")
start_time = time.time()
berhasil = 0

for i, data in enumerate(fase3_dataset):
    try:
        master_path = data.get("master_ktp_path", "")
        if not os.path.exists(master_path):
            print(f"   ⚠️ Data ke-{i+1}: File master '{master_path}' hilang, skip.")
            continue
            
        geo_config = random.choice(template_configs)
        template_path = geo_config.get("template_path")
        
        if not os.path.exists(template_path):
            print(f"   ⚠️ Template '{template_path}' tidak ditemukan, skip...")
            continue
            
        # 3. BACA GAMBAR (Pertahankan Channel Alpha/Transparansi)
        # IMREAD_UNCHANGED memastikan 4 channel (B, G, R, Alpha) dimuat
        img_master = cv2.imread(master_path, cv2.IMREAD_UNCHANGED) 
        img_clean = cv2.imread(template_path) # 3 channel (B, G, R)
        
        mh, mw = img_master.shape[:2]
        ch, cw = img_clean.shape[:2]
        
        # 4. HOMOGRAFI PERSPEKTIF 3D
        src_pts = np.array([[0, 0], [mw - 1, 0], [mw - 1, mh - 1], [0, mh - 1]], dtype=np.float32)
        dst_pts = np.array(geo_config["ktp_corners"], dtype=np.float32)
        
        M = cv2.getPerspectiveTransform(src_pts, dst_pts)
        
        # Warp KTP transparan (tetap 4 channel), area kosong akan ber-alpha 0
        warped_ktp_bgra = cv2.warpPerspective(img_master, M, (cw, ch), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_CONSTANT, borderValue=(0,0,0,0))
        
        # Pisahkan warna (BGR) dan Transparansi (Alpha)
        warped_bgr = warped_ktp_bgra[:, :, :3]
        alpha_channel = warped_ktp_bgra[:, :, 3].copy()
        
        # 5. PUNCH HOLE (BOLONGI ALPHA CHANNEL DENGAN POLIGON JARI)
        if "hand_mask" in geo_config and geo_config["hand_mask"]:
            saved_masks = geo_config["hand_mask"]
            if len(saved_masks) > 0 and isinstance(saved_masks[0][0], (int, float)):
                # Mode 1 jari (Lama)
                cv2.fillPoly(alpha_channel, [np.array(saved_masks, dtype=np.int32)], 0)
            else:
                # Mode Multi-jari (Baru)
                for poly in saved_masks:
                    if len(poly) >= 3:
                        # Angka 0 artinya transparansi 100% (Bolong)
                        cv2.fillPoly(alpha_channel, [np.array(poly, dtype=np.int32)], 0)
                        
        # 6. ALPHA BLENDING (Compositing KTP ke Background)
        # Normalisasi alpha ke skala 0.0 - 1.0
        alpha_mask = alpha_channel.astype(float) / 255.0
        alpha_mask = np.expand_dims(alpha_mask, axis=2) # Jadikan 3D untuk dikali dengan BGR
        
        img_clean_float = img_clean.astype(float)
        warped_bgr_float = warped_bgr.astype(float)
        
        # Rumus magis Photoshop: (Foreground * Alpha) + (Background * (1 - Alpha))
        final_float = (warped_bgr_float * alpha_mask) + (img_clean_float * (1.0 - alpha_mask))
        final_composite = np.clip(final_float, 0, 255).astype(np.uint8)
        
        # 7. PENYIMPANAN
        nik_file = data.get("nik", f"RECONSTRUCTED_{i}")
        output_filename = f"real_ktp_{nik_file}.jpg"
        output_path = os.path.join(DIR_FASE3_2, output_filename)
        
        # Simpan JPG kualitas 95% (karena hasil akhir sudah menyatu dengan background, tidak butuh transparan lagi)
        cv2.imwrite(output_path, final_composite, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
        
        # Update metadata JSON
        data["warped_ktp_path"] = output_path
        data["used_template"] = geo_config.get("template_name")
        berhasil += 1
        
        if (i + 1) % 10 == 0:
            print(f"   [+] Berhasil melipat, masking & blending {i + 1}/100 KTP Fotorealistis...")
            
    except Exception as e:
        print(f"❌ Error pada indeks ke-{i} (NIK: {data.get('nik')}): {e}")

# 8. SIMPAN JSON LANJUTAN
with open(FILE_FASE3_2, "w", encoding="utf-8") as f:
    json.dump(fase3_dataset, f, indent=4, ensure_ascii=False)

end_time = time.time()
print("="*50)
print(f"✅ FASE 3.2 SELESAI ({berhasil}/{len(fase3_dataset)} sukses diproses)")
print(f"Waktu Komputasi: {end_time - start_time:.2f} detik.")
print(f"Dataset Fotorealistis Tersimpan di  : {DIR_FASE3_2}")
print(f"File Estafet JSON Lanjutan          : {FILE_FASE3_2}")
print("="*50)

🔄 Memuat data rekonstruksi...
✅ Terdeteksi 100 dataset KTP Master transparan.
✅ Terdeteksi 10 konfigurasi Template Clean siap pakai.

🚀 Memulai orkestrasi Homografi & Alpha Blending untuk 100 data...
   [+] Berhasil melipat, masking & blending 10/100 KTP Fotorealistis...
   [+] Berhasil melipat, masking & blending 20/100 KTP Fotorealistis...
   [+] Berhasil melipat, masking & blending 30/100 KTP Fotorealistis...
   [+] Berhasil melipat, masking & blending 40/100 KTP Fotorealistis...
   [+] Berhasil melipat, masking & blending 50/100 KTP Fotorealistis...
   [+] Berhasil melipat, masking & blending 60/100 KTP Fotorealistis...
   [+] Berhasil melipat, masking & blending 70/100 KTP Fotorealistis...
   [+] Berhasil melipat, masking & blending 80/100 KTP Fotorealistis...
   [+] Berhasil melipat, masking & blending 90/100 KTP Fotorealistis...
   [+] Berhasil melipat, masking & blending 100/100 KTP Fotorealistis...
✅ FASE 3.2 SELESAI (100/100 sukses diproses)
Waktu Komputasi: 23.38 detik.
Data

# Fase 4 - Scenario Distribution & Background Generator (GAN)

In [51]:
# ==========================================
# FASE 4: SKENARIO DISTRIBUSI & GAN BACKGROUND
# ==========================================
# Asumsi: Master Configuration sudah dijalankan di cell atas
# Variabel yang dipakai: DIR_FASE4, FILE_FASE1, FILE_FASE2, FILE_FASE3, FILE_FASE4

print("🔄 Memeriksa dan memuat data dari fase sebelumnya...")

# 1. LOAD DATA IDENTITAS (Cari yang paling lengkap: Fase 2 -> Fase 1)
if os.path.exists(FILE_FASE2):
    print(f"✅ Menemukan data dengan wajah: {FILE_FASE2}")
    with open(FILE_FASE2, "r", encoding="utf-8") as f:
        final_datasets = json.load(f)
elif os.path.exists(FILE_FASE1):
    print(f"✅ Menemukan data teks saja: {FILE_FASE1}")
    with open(FILE_FASE1, "r", encoding="utf-8") as f:
        final_datasets = json.load(f)
else:
    raise FileNotFoundError("❌ Data dari Fase 1 atau Fase 2 tidak ditemukan!")

# 2. LOAD DATA KALIBRASI TEMPLATE (Fase 3)
if not os.path.exists(FILE_FASE3):
    raise FileNotFoundError(f"❌ File konfigurasi {FILE_FASE3} tidak ditemukan. Jalankan Fase 3 terlebih dahulu.")

with open(FILE_FASE3, "r", encoding="utf-8") as f:
    template_configs = json.load(f)

print(f"✓ Berhasil load {len(final_datasets)} data identitas dan {len(template_configs)} template KTP.")

# ==========================================
# FUNGSI PROMPT TEKSTUR BACKGROUND
# ==========================================
def get_random_texture_prompt():
    """Memberikan prompt acak untuk background permukaan KTP."""
    textures = [
        "a top down view of a rustic wooden table texture",
        "a close up of an asphalt road surface",
        "a top down view of a black leather wallet texture",
        "a close up of messy white bed sheet fabric",
        "a flat lay of a clean marble floor tile",
        "a top down view of a denim jeans fabric",
        "a close up of a rusty metal table surface",
        "a top down view of an office desk pad"
    ]
    prompt = random.choice(textures) + ", photorealistic, highly detailed, 8k resolution, flat lighting, macro photography"
    negative_prompt = "objects, blurry, low resolution, text, watermark, deformed, depth of field (too much blur)"
    return prompt, negative_prompt

# ==========================================
# GENERATE 90 BACKGROUND GAN
# ==========================================
print("\n🚀 Memulai Generasi 90 Background Tekstur untuk Skenario B...")
generated_bgs = []
start_time = time.time()

for i in range(90):
    # UPDATE: Gambar background disimpan di folder Fase 4
    bg_path = os.path.join(DIR_FASE4, f"bg_gan_{i:03d}.jpg")
    
    # [FITUR RESUME] Jika file sudah ada, tidak perlu di-generate lagi
    if not os.path.exists(bg_path):
        prompt, neg_prompt = get_random_texture_prompt()
        print(f"[{i+1}/90] Merender tekstur: {prompt.split(',')[0]}...")
        
        # Render background (pastikan 'pipe' sudah terinisialisasi di memori dari eksekusi Fase 2)
        try:
            image = pipe(prompt=prompt, negative_prompt=neg_prompt, num_inference_steps=20).images[0]
            image.save(bg_path)
        except NameError:
            print("❌ Error: Variabel 'pipe' (Stable Diffusion) tidak ditemukan. Pastikan kamu sudah run cell inisialisasi modelnya.")
            break
            
    generated_bgs.append(bg_path)

print(f"✅ Selesai menyiapkan 90 Background GAN dalam {(time.time() - start_time) / 60:.2f} menit.")

# ==========================================
# PEMETAAN TEMPLATE & SKENARIO
# ==========================================
print("\n🔄 Memetakan Data ke Template dan Skenario...")

# Menggandakan 10 template menjadi 100
templates_100 = template_configs * 10
random.shuffle(templates_100) # Acak urutan agar distribusinya merata

for i, data in enumerate(final_datasets):
    # Tempelkan konfigurasi template 8-titik ke data identitas
    data["template"] = templates_100[i]
    
    # Aturan Distribusi Skenario
    if i < 10:
        data["skenario"] = "A" # 10 Data pertama mempertahankan background aslinya
        data["background_gan"] = None
    else:
        data["skenario"] = "B" # 90 Data sisanya akan diganti background-nya
        # Ambil background GAN sesuai indeks (i - 10 karena mulai dari indeks 10)
        data["background_gan"] = generated_bgs[i - 10]

# ==========================================
# SIMPAN DATA FASE 4
# ==========================================
with open(FILE_FASE4, "w", encoding="utf-8") as f:
    json.dump(final_datasets, f, indent=4, ensure_ascii=False)

print("="*50)
print(f"✅ FASE 4 SELESAI")
print(f"Pemetaan Skenario tersimpan dengan aman di: {FILE_FASE4}")
print(f"90 Background GAN tersimpan di folder: {DIR_FASE4}")
print("="*50)

🔄 Memeriksa dan memuat data dari fase sebelumnya...
✅ Menemukan data dengan wajah: Fase2-Output\data_fase2.json
✓ Berhasil load 100 data identitas dan 10 template KTP.

🚀 Memulai Generasi 90 Background Tekstur untuk Skenario B...
✅ Selesai menyiapkan 90 Background GAN dalam 0.00 menit.

🔄 Memetakan Data ke Template dan Skenario...
✅ FASE 4 SELESAI
Pemetaan Skenario tersimpan dengan aman di: Fase4-Output\data_fase4.json
90 Background GAN tersimpan di folder: Fase4-Output


In [4]:
import os
import json

# ==========================================
# FASE 4: SKENARIO DISTRIBUSI (BYPASS RENDER)
# ==========================================
# Mengambil estafet dari Fase 3.2 (Bukan Fase 2!)
FILE_FASE3_2 = os.path.join("Fase3.2-Output", "data_fase3_2.json")
DIR_FASE4 = "Fase4-Output"
FILE_FASE4 = os.path.join(DIR_FASE4, "data_fase4.json")

print("🔄 Menghubungkan data fotorealistis dari Fase 3.2 ke Fase 4...")

if not os.path.exists(FILE_FASE3_2):
    raise FileNotFoundError(f"❌ File {FILE_FASE3_2} tidak ditemukan!")

with open(FILE_FASE3_2, "r", encoding="utf-8") as f:
    dataset_fase4 = json.load(f)

# Mengumpulkan 90 Background GAN yang sudah kamu buat sebelumnya
generated_bgs = []
for i in range(90):
    bg_path = os.path.join(DIR_FASE4, f"bg_gan_{i:03d}.jpg")
    if os.path.exists(bg_path):
        generated_bgs.append(bg_path)

print(f"✅ Ditemukan {len(generated_bgs)} Background GAN yang sudah jadi.")

print("\n🔄 Memetakan Skenario A (Asli) & B (GAN)...")

# Distribusi Skenario
for i, data in enumerate(dataset_fase4):
    if i < 10:
        data["skenario"] = "A" # KTP tetap di background asli template clean
        data["background_gan"] = None
    else:
        data["skenario"] = "B" # KTP akan dipotong dan dipindah ke background GAN
        # Menggunakan sisa pembagian (modulo) agar aman dari IndexError
        bg_index = (i - 10) % len(generated_bgs) if len(generated_bgs) > 0 else 0
        data["background_gan"] = generated_bgs[bg_index] if len(generated_bgs) > 0 else None

# Simpan hasil akhir estafet
with open(FILE_FASE4, "w", encoding="utf-8") as f:
    json.dump(dataset_fase4, f, indent=4, ensure_ascii=False)

print("="*50)
print("✅ REVISI FASE 4 SELESAI (Tanpa Render Ulang)")
print(f"Data JSON yang benar tersimpan di: {FILE_FASE4}")
print("="*50)

🔄 Menghubungkan data fotorealistis dari Fase 3.2 ke Fase 4...
✅ Ditemukan 90 Background GAN yang sudah jadi.

🔄 Memetakan Skenario A (Asli) & B (GAN)...
✅ REVISI FASE 4 SELESAI (Tanpa Render Ulang)
Data JSON yang benar tersimpan di: Fase4-Output\data_fase4.json


# Fase 5 - BACKGROUND REPLACEMENT & AUGMENTATION


In [9]:
import os
import json
import cv2
import time
import numpy as np
import shutil

# ==========================================
# FASE 5: BACKGROUND REPLACEMENT (ANTI-OUTLINE FIX)
# ==========================================
FILE_FASE4 = os.path.join("Fase4-Output", "data_fase4.json")
DIR_FASE5 = "Fase5-Output"
FILE_FASE5 = os.path.join(DIR_FASE5, "data_fase5.json")
CONFIG_TEMPLATE = os.path.join("Fase3.1-Output", "config_clean_templates.json")

if not os.path.exists(DIR_FASE5):
    os.makedirs(DIR_FASE5)

print("🔄 Memuat data pemetaan dan geometri...")
if not os.path.exists(FILE_FASE4): raise FileNotFoundError(f"❌ File {FILE_FASE4} tidak ditemukan!")
if not os.path.exists(CONFIG_TEMPLATE): raise FileNotFoundError(f"❌ File {CONFIG_TEMPLATE} tidak ditemukan!")

with open(FILE_FASE4, "r", encoding="utf-8") as f:
    final_datasets = json.load(f)
with open(CONFIG_TEMPLATE, "r", encoding="utf-8") as f:
    template_configs = json.load(f)

print(f"🚀 Memulai Finalisasi Komposisi (Scenario Distribution) untuk {len(final_datasets)} data...")
start_time = time.time()
berhasil = 0

TINGKAT_EROSI = 2

for i, data in enumerate(final_datasets):
    try:
        real_ktp_path = data.get("warped_ktp_path")
        if not real_ktp_path or not os.path.exists(real_ktp_path):
            continue

        nik_file = data.get("nik", f"UNKNOWN_{i}")
        output_filename = f"final_ktp_{nik_file}.jpg"
        output_path = os.path.join(DIR_FASE5, output_filename)
        skenario = data.get("skenario", "A")

        if skenario == "A":
            shutil.copy2(real_ktp_path, output_path)
            data["final_image_path"] = output_path
            berhasil += 1

        elif skenario == "B":
            used_template_name = data.get("used_template")
            geo_config = next((t for t in template_configs if t["template_name"] == used_template_name), None)
            
            if not geo_config: continue

            bg_gan_path = data.get("background_gan")
            if not bg_gan_path or not os.path.exists(bg_gan_path):
                shutil.copy2(real_ktp_path, output_path)
                data["final_image_path"] = output_path
                berhasil += 1
                continue

            img_real = cv2.imread(real_ktp_path)
            img_gan = cv2.imread(bg_gan_path)
            h_real, w_real = img_real.shape[:2]
            img_gan_resized = cv2.resize(img_gan, (w_real, h_real), interpolation=cv2.INTER_CUBIC)

            # --- BUAT MASK KTP & TANGAN ---
            mask_fg = np.zeros((h_real, w_real), dtype=np.uint8)
            
            ktp_pts = np.array(geo_config["ktp_corners"], dtype=np.int32)
            cv2.fillPoly(mask_fg, [ktp_pts], 255)
            
            if "hand_mask" in geo_config and geo_config["hand_mask"]:
                saved_masks = geo_config["hand_mask"]
                if len(saved_masks) > 0 and isinstance(saved_masks[0][0], (int, float)):
                    cv2.fillPoly(mask_fg, [np.array(saved_masks, dtype=np.int32)], 255)
                else:
                    for poly in saved_masks:
                        if len(poly) >= 3:
                            cv2.fillPoly(mask_fg, [np.array(poly, dtype=np.int32)], 255)

            # --- REVISI ANTI-OUTLINE ---
            kernel_erode = np.ones((3,3), np.uint8)
            mask_eroded = cv2.erode(mask_fg, kernel_erode, iterations=TINGKAT_EROSI)
            mask_soft = cv2.GaussianBlur(mask_eroded, (3, 3), 0)
            
            # --- FIX: NUMPY ALPHA BLENDING ---
            # Jadikan mask berukuran 3D: (Height, Width, 1) agar kompatibel dengan gambar BGR
            alpha = mask_soft.astype(float) / 255.0
            alpha = np.expand_dims(alpha, axis=2) 
            
            # Konversi gambar ke float
            img_real_float = img_real.astype(float)
            img_gan_float = img_gan_resized.astype(float)
            
            # Matematika Blending Numpy
            composite_float = (img_real_float * alpha) + (img_gan_float * (1.0 - alpha))
            final_composite = np.clip(composite_float, 0, 255).astype(np.uint8)

            cv2.imwrite(output_path, final_composite, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
            data["final_image_path"] = output_path
            berhasil += 1

        if (i + 1) % 10 == 0:
            print(f"   [+] Menyelesaikan komposisi antib-outline {i + 1}/100 data...")

    except Exception as e:
        print(f"❌ Error pada dataset ke-{i}: {e}")

with open(FILE_FASE5, "w", encoding="utf-8") as f:
    json.dump(final_datasets, f, indent=4, ensure_ascii=False)

end_time = time.time()
print("="*50)
print(f"✅ FASE 5 (FINALISASI ANTIB-OUTLINE) SELESAI ({berhasil}/{len(final_datasets)} sukses)")
print(f"KTP Final Fotorealistis tersimpan di: {DIR_FASE5}")
print("="*50)

🔄 Memuat data pemetaan dan geometri...
🚀 Memulai Finalisasi Komposisi (Scenario Distribution) untuk 100 data...
   [+] Menyelesaikan komposisi antib-outline 10/100 data...
   [+] Menyelesaikan komposisi antib-outline 20/100 data...
   [+] Menyelesaikan komposisi antib-outline 30/100 data...
   [+] Menyelesaikan komposisi antib-outline 40/100 data...
   [+] Menyelesaikan komposisi antib-outline 50/100 data...
   [+] Menyelesaikan komposisi antib-outline 60/100 data...
   [+] Menyelesaikan komposisi antib-outline 70/100 data...
   [+] Menyelesaikan komposisi antib-outline 80/100 data...
   [+] Menyelesaikan komposisi antib-outline 90/100 data...
   [+] Menyelesaikan komposisi antib-outline 100/100 data...
✅ FASE 5 (FINALISASI ANTIB-OUTLINE) SELESAI (100/100 sukses)
KTP Final Fotorealistis tersimpan di: Fase5-Output


# Fase 6 - Compositing & Visual Harmonization

In [10]:
import cv2
import json
import numpy as np
import os
import time
import shutil

# ==========================================
# FASE 6: HARMONISASI VISUAL (REINHARD & SHADOW)
# ==========================================
FILE_FASE5 = os.path.join("Fase5-Output", "data_fase5.json")
DIR_FASE6 = "Fase6-Output"
FILE_FASE6 = os.path.join(DIR_FASE6, "data_fase6.json")
CONFIG_TEMPLATE = os.path.join("Fase3.1-Output", "config_clean_templates.json")

if not os.path.exists(DIR_FASE6):
    os.makedirs(DIR_FASE6)

print("🔄 Memuat data dasar dari Fase 5...")
if not os.path.exists(FILE_FASE5): raise FileNotFoundError(f"❌ File {FILE_FASE5} tidak ditemukan.")
if not os.path.exists(CONFIG_TEMPLATE): raise FileNotFoundError(f"❌ File {CONFIG_TEMPLATE} tidak ditemukan.")

with open(FILE_FASE5, "r", encoding="utf-8") as f:
    final_datasets = json.load(f)
with open(CONFIG_TEMPLATE, "r", encoding="utf-8") as f:
    template_configs = json.load(f)

# ==========================================
# FUNGSI HARMONISASI TINGKAT LANJUT
# ==========================================
def apply_reinhard_color_transfer(source, target, mask_2d, transfer_ratio=0.2):
    """Menyamakan pencahayaan dan tone warna (Reinhard Method)."""
    src_lab = cv2.cvtColor(source, cv2.COLOR_BGR2LAB).astype("float32")
    tgt_lab = cv2.cvtColor(target, cv2.COLOR_BGR2LAB).astype("float32")
    
    mask_bool = mask_2d > 128
    
    for i in range(3):
        src_l = src_lab[:, :, i]
        tgt_l = tgt_lab[:, :, i]
        
        # Hindari pembagian dengan nol jika mask kosong
        if not mask_bool.any():
            continue
            
        src_mean, src_std = src_l[mask_bool].mean(), src_l[mask_bool].std()
        tgt_mean, tgt_std = tgt_l.mean(), tgt_l.std()
        
        transferred = ((src_l[mask_bool] - src_mean) * (tgt_std / (src_std + 1e-5))) + tgt_mean
        src_lab[:, :, i][mask_bool] = (1 - transfer_ratio) * src_l[mask_bool] + transfer_ratio * transferred
        
    src_lab = np.clip(src_lab, 0, 255).astype("uint8")
    return cv2.cvtColor(src_lab, cv2.COLOR_LAB2BGR)

def generate_drop_shadow(bg_img, mask_2d, shift_x=12, shift_y=18, blur_ksize=(35, 35), opacity=0.55):
    """Membuat bayangan dinamis di bawah KTP dan Jari."""
    h, w = mask_2d.shape
    M = np.float32([[1, 0, shift_x], [0, 1, shift_y]])
    shifted_mask = cv2.warpAffine(mask_2d, M, (w, h))
    
    blurred_shadow = cv2.GaussianBlur(shifted_mask, blur_ksize, 0)
    shadow_alpha = (blurred_shadow / 255.0) * opacity
    
    res_bg = bg_img.astype(float).copy()
    for c in range(3):
        res_bg[:, :, c] = res_bg[:, :, c] * (1.0 - shadow_alpha)
        
    return np.clip(res_bg, 0, 255).astype(np.uint8)

# ==========================================
# EKSEKUSI COMPOSITING
# ==========================================
print("🚀 Memulai Fase 6: Compositing & Harmonisasi Reinhard untuk 100 data...")
start_time = time.time()
berhasil = 0
TINGKAT_EROSI = 2

for i, data in enumerate(final_datasets):
    try:
        nik_file = data.get("nik", f"UNKNOWN_{i}")
        output_filename = f"cinematic_ktp_{nik_file}.jpg"
        output_path = os.path.join(DIR_FASE6, output_filename)
        skenario = data.get("skenario", "A")

        if skenario == "A":
            # Skenario A tidak butuh harmonisasi karena sudah berada di background aslinya
            # Kita cukup ambil output Fase 5
            img_path = data.get("final_image_path")
            if os.path.exists(img_path):
                shutil.copy2(img_path, output_path)
            data["cinematic_image_path"] = output_path
            berhasil += 1

        elif skenario == "B":
            # Ambil bahan baku mentah dari Fase 3.2 (KTP yang sudah di-warp & di-masking jari)
            real_ktp_path = data.get("warped_ktp_path")
            bg_gan_path = data.get("background_gan")
            used_template_name = data.get("used_template")
            
            if not real_ktp_path or not bg_gan_path: continue
            
            geo_config = next((t for t in template_configs if t["template_name"] == used_template_name), None)
            if not geo_config: continue

            img_real = cv2.imread(real_ktp_path)
            img_gan = cv2.imread(bg_gan_path)
            h_real, w_real = img_real.shape[:2]
            
            # Sesuaikan ukuran GAN
            img_gan_resized = cv2.resize(img_gan, (w_real, h_real), interpolation=cv2.INTER_CUBIC)

            # --- 1. REKONSTRUKSI MASK (KTP + TANGAN) ---
            mask_fg = np.zeros((h_real, w_real), dtype=np.uint8)
            cv2.fillPoly(mask_fg, [np.array(geo_config["ktp_corners"], dtype=np.int32)], 255)
            
            if "hand_mask" in geo_config and geo_config["hand_mask"]:
                saved_masks = geo_config["hand_mask"]
                if len(saved_masks) > 0 and isinstance(saved_masks[0][0], (int, float)):
                    cv2.fillPoly(mask_fg, [np.array(saved_masks, dtype=np.int32)], 255)
                else:
                    for poly in saved_masks:
                        if len(poly) >= 3:
                            cv2.fillPoly(mask_fg, [np.array(poly, dtype=np.int32)], 255)

            # Erosi untuk mencegah Halo Effect (Outline hitam)
            kernel_erode = np.ones((3,3), np.uint8)
            mask_eroded = cv2.erode(mask_fg, kernel_erode, iterations=TINGKAT_EROSI)
            mask_soft = cv2.GaussianBlur(mask_eroded, (3, 3), 0)

            # --- 2. HARMONISASI WARNA REINHARD ---
            # Kita ubah warna img_real (KTP asli) agar menyerap ambient color dari img_gan_resized
            img_real_harmonized = apply_reinhard_color_transfer(img_real, img_gan_resized, mask_fg, transfer_ratio=0.15)

            # --- 3. GENERATE DROP SHADOW ---
            # Kita tembakkan bayangan mask ke atas meja kayu GAN
            bg_gan_shadowed = generate_drop_shadow(img_gan_resized, mask_soft, shift_x=12, shift_y=18, opacity=0.6)

            # --- 4. FINAL ALPHA BLENDING ---
            alpha = np.expand_dims(mask_soft.astype(float) / 255.0, axis=2)
            fg_float = img_real_harmonized.astype(float)
            bg_float = bg_gan_shadowed.astype(float)
            
            final_composite = np.clip((fg_float * alpha) + (bg_float * (1.0 - alpha)), 0, 255).astype(np.uint8)

            cv2.imwrite(output_path, final_composite, [int(cv2.IMWRITE_JPEG_QUALITY), 97])
            data["cinematic_image_path"] = output_path
            berhasil += 1

        if (i + 1) % 10 == 0:
            print(f"   [+] Harmonisasi cinematic selesai untuk {i + 1}/100 data...")

    except Exception as e:
        print(f"❌ Error pada dataset ke-{i}: {e}")

# ==========================================
# SIMPAN DATA ESTAFET FASE 6
# ==========================================
with open(FILE_FASE6, "w", encoding="utf-8") as f:
    json.dump(final_datasets, f, indent=4, ensure_ascii=False)

print("="*50)
print(f"✅ FASE 6 SELESAI ({berhasil}/{len(final_datasets)} berhasil diproses)")
print(f"Waktu Eksekusi: {time.time() - start_time:.2f} detik.")
print(f"KTP Cinematic tersimpan di folder: {DIR_FASE6}")
print(f"Data estafet tersimpan di file   : {FILE_FASE6}")
print("="*50)

🔄 Memuat data dasar dari Fase 5...
🚀 Memulai Fase 6: Compositing & Harmonisasi Reinhard untuk 100 data...
   [+] Harmonisasi cinematic selesai untuk 10/100 data...
   [+] Harmonisasi cinematic selesai untuk 20/100 data...
   [+] Harmonisasi cinematic selesai untuk 30/100 data...
   [+] Harmonisasi cinematic selesai untuk 40/100 data...
   [+] Harmonisasi cinematic selesai untuk 50/100 data...
   [+] Harmonisasi cinematic selesai untuk 60/100 data...
   [+] Harmonisasi cinematic selesai untuk 70/100 data...
   [+] Harmonisasi cinematic selesai untuk 80/100 data...
   [+] Harmonisasi cinematic selesai untuk 90/100 data...
   [+] Harmonisasi cinematic selesai untuk 100/100 data...
✅ FASE 6 SELESAI (100/100 berhasil diproses)
Waktu Eksekusi: 46.72 detik.
KTP Cinematic tersimpan di folder: Fase6-Output
Data estafet tersimpan di file   : Fase6-Output\data_fase6.json


# Fase 7 - Finalization

In [ ]:
# ==========================================
# FASE 7: FINALISASI & GROUND TRUTH ML
# ==========================================
# Asumsi: Master Configuration sudah dijalankan di cell paling atas
# Variabel yang dipakai: DIR_FASE7, FILE_FASE6, FILE_GROUND_TRUTH

print("🔄 Memuat data KTP composited dari Fase 6...")
if not os.path.exists(FILE_FASE6):
    raise FileNotFoundError(f"❌ File {FILE_FASE6} tidak ditemukan. Jalankan Fase 6 terlebih dahulu.")

with open(FILE_FASE6, "r", encoding="utf-8") as f:
    final_datasets = json.load(f)

# ==========================================
# FUNGSI SIMULASI KAMERA FISIK
# ==========================================
def apply_camera_physics(image):
    # 1. SLIGHT LENS BLUR (Fokus Lensa Sedikit Melunak)
    # Menggunakan kernel 3x3, sangat halus agar teks KTP tetap terbaca (OCR-friendly)
    blurred = cv2.GaussianBlur(image, (3, 3), 0)
    
    # 2. GLOBAL SENSOR NOISE (Bintik ISO Kamera)
    row, col, ch = blurred.shape
    mean = 0
    var = random.uniform(10, 25) # Variasi intensitas noise antar foto
    sigma = var ** 0.5
    
    # Generate matriks noise dan tambahkan ke gambar
    gauss_noise = np.random.normal(mean, sigma, (row, col, ch))
    gauss_noise = gauss_noise.reshape(row, col, ch)
    noisy_img = blurred + gauss_noise
    
    # Kliping nilai pixel agar tetap di rentang 0-255
    noisy_img = np.clip(noisy_img, 0, 255).astype(np.uint8)
    
    # 3. JPEG COMPRESSION ARTIFACTS
    # Mensimulasikan gambar yang dikirim via WhatsApp (kualitas 75 - 90)
    encode_param = [int(cv2.IMWRITE_JPEG_QUALITY), random.randint(75, 90)]
    result, encimg = cv2.imencode('.jpg', noisy_img, encode_param)
    final_img = cv2.imdecode(encimg, 1)
    
    return final_img

# ==========================================
# EKSEKUSI FINALISASI
# ==========================================
print(f"🚀 Memulai Fase 7: Penambahan Efek Kamera & Pembuatan Ground Truth...")
start_time = time.time()
berhasil = 0

# List untuk menyimpan label ground truth untuk model ML-mu
ml_ground_truth = []

for i, data in enumerate(final_datasets):
    try:
        # Load gambar hasil compositing dari Fase 6
        composited_path = data.get("composited_path")
        if not composited_path or not os.path.exists(composited_path):
            print(f"⚠️ Melewati data ke-{i}: Gambar composited tidak ditemukan di {composited_path}.")
            continue
            
        img = cv2.imread(composited_path)
        
        # Terapkan efek kamera
        final_output = apply_camera_physics(img)
        
        # Nama file final menggunakan NIK untuk mempermudah indexing
        nik_file = data.get('nik', f"UNKNOWN_{i}")
        final_filename = f"ktp_synthetic_{nik_file}.jpg"
        
        # UPDATE: Simpan gambar ke folder Fase 7
        final_path = os.path.join(DIR_FASE7, final_filename)
        cv2.imwrite(final_path, final_output)
        
        # ==========================================
        # GROUND TRUTH LENGKAP UNTUK ML
        # ==========================================
        clean_label = {
            "file_name": final_filename,
            "skenario_augmentasi": data.get("skenario", "B"),
            "data_teks": {
                "provinsi": data.get("provinsi", ""),
                "kota_kab": data.get("kota_kab", ""),
                "nik": data.get("nik", ""),
                "nama": data.get("nama", ""),
                "tempat_lahir": data.get("tempat_lahir", ""),
                "tgl_lahir": data.get("tgl_lahir", ""),
                "jenis_kelamin": data.get("jenis_kelamin", ""),
                "gol_darah": data.get("gol_darah", ""),
                "alamat": data.get("alamat", ""),
                "rt_rw": data.get("rt_rw", ""),
                "kel_desa": data.get("kel_desa", ""),
                "kecamatan": data.get("kecamatan", ""),
                "agama": data.get("agama", ""),
                "status_perkawinan": data.get("status_perkawinan", ""),
                "pekerjaan": data.get("pekerjaan", ""),
                "kewarganegaraan": data.get("kewarganegaraan", ""),
                "berlaku_hingga": data.get("berlaku_hingga", ""),
                "tgl_pembuatan": data.get("tgl_pembuatan", "")
            },
            "bounding_boxes": {
                "koordinat_kartu_ktp": data.get("template", {}).get("card_corners", []),
                "koordinat_pas_foto": data.get("template", {}).get("photo_corners", [])
            }
        }
        ml_ground_truth.append(clean_label)
        
        berhasil += 1
        if (i + 1) % 10 == 0:
            print(f"   Finalisasi {i + 1}/100 dataset selesai...")
            
    except Exception as e:
        print(f"❌ Error pada dataset ke-{i}: {e}")

# ==========================================
# SIMPAN GROUND TRUTH LENGKAP
# ==========================================
with open(FILE_GROUND_TRUTH, "w", encoding="utf-8") as f:
    json.dump(ml_ground_truth, f, indent=4, ensure_ascii=False)

print("="*50)
print(f"🎉 SELURUH WORKFLOW SELESAI! 🎉")
print(f"Berhasil merender {berhasil} dataset KTP Sintetik yang siap di-training.")
print(f"Waktu Eksekusi: {time.time() - start_time:.2f} detik.")
print(f"\n📁 HASIL AKHIR:")
print(f"1. Gambar (Dataset Utama): Folder '{DIR_FASE7}'")
print(f"2. Label ML Lengkap      : File '{FILE_GROUND_TRUTH}'")
print("="*50)